
# # LunarXReg: Chandrayaan-2 Multi-Modal Image Registration MVP
#
# **Problem Statement ID**: 26166
#
# **Title**: Multi-modal, Sun angle and scale invariant image correspondence using Chandrayaan-2 optical images (OHRC, TMC-2 and IIRS)
#
# **Honest Limitations**:
# - Real registration success requires physical overlap between source and reference images.
# - Images from different dates/sensors may not overlap.
# - Synthetic validation mode is included to validate the registration engine on known transformations.
# - Output confidence is evidence-based, not a guarantee.
# - No mission-grade or guaranteed sub-pixel accuracy is claimed.
# - Raw product parsing may fail; browse preview fallback is used with clear diagnostics.
# - GeoTIFF is produced only if valid CRS/geotransform is available; otherwise outputs are in image space.
# - Matches are never fabricated. REVIEW or REJECT is output when confidence is inadequate.

#  CELL GROUP B — Environment Setup

In [1]:
import os
import sys
import json
import random
import logging
import warnings
import traceback
import subprocess
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple, Dict, Any, List, Union

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception as e:
    print(f"Not running in Colab or Drive mount failed: {e}")
    IN_COLAB = False

# Install required packages
packages = [
    "opencv-contrib-python-headless",
    "spectral",
    "rasterio",
    "tifffile",
    "pandas",
    "scikit-image",
    "matplotlib",
    "seaborn",
    "lxml",
    "tqdm",
]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_").split("==")[0])
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Import after installation
import cv2
import spectral
import rasterio
import tifffile
from skimage import exposure, transform as sktransform, feature as skfeature
from lxml import etree as lxml_etree
import xml.etree.ElementTree as ET

# GPU detection
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    if GPU_AVAILABLE:
        print(f"GPU available: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU not available. Using CPU-only implementation.")
except ImportError:
    GPU_AVAILABLE = False
    print("PyTorch not installed. Using CPU-only implementation.")

# Deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    torch.manual_seed(SEED)
    if GPU_AVAILABLE:
        torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# Paths
ROOT_DIR = Path("/content/drive/MyDrive/26166_DATA/raw")
RESULTS_DIR = Path("/content/drive/MyDrive/26166_DATA/results")

# Create output directories
OUTPUT_SUBDIRS = [
    "previews",
    "metadata",
    "structural_views",
    "matches",
    "registered",
    "reports",
    "synthetic_validation",
    "logs",
]
for subdir in OUTPUT_SUBDIRS:
    (RESULTS_DIR / subdir).mkdir(parents=True, exist_ok=True)

# Logging setup
LOG_FILE = RESULTS_DIR / "logs" / "lunarxreg.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="a"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("LunarXReg")
logger.info("=" * 80)
logger.info("LunarXReg session started")
logger.info(f"ROOT_DIR: {ROOT_DIR}")
logger.info(f"RESULTS_DIR: {RESULTS_DIR}")
logger.info(f"GPU available: {GPU_AVAILABLE}")
logger.info("=" * 80)

# Discover files
def discover_files() -> pd.DataFrame:
    """Discover all relevant files in the raw data directory."""
    records = []
    if not ROOT_DIR.exists():
        logger.error(f"ROOT_DIR does not exist: {ROOT_DIR}")
        return pd.DataFrame()

    for sensor_dir in sorted(ROOT_DIR.iterdir()):
        if not sensor_dir.is_dir():
            continue
        sensor = sensor_dir.name.lower()
        # Find all XML files
        for xml_path in sensor_dir.rglob("*.xml"):
            rel = xml_path.relative_to(ROOT_DIR)
            parts = rel.parts
            product_type = parts[1] if len(parts) > 1 else "unknown"
            # Find corresponding data files
            base = xml_path.stem
            parent = xml_path.parent
            img_path = None
            hdr_path = None
            qub_path = None
            csv_path = None
            browse_png = None

            # Look for .img, .hdr, .qub, .csv, .png in same or nearby directories
            for candidate in parent.iterdir():
                if candidate.suffix.lower() == ".img" and candidate.stem == base:
                    img_path = candidate
                elif candidate.suffix.lower() == ".hdr" and candidate.stem == base:
                    hdr_path = candidate
                elif candidate.suffix.lower() == ".qub" and candidate.stem == base:
                    qub_path = candidate
                elif candidate.suffix.lower() == ".csv" and base in candidate.stem:
                    csv_path = candidate
                elif candidate.suffix.lower() == ".png" and base in candidate.stem:
                    browse_png = candidate

            # Also check for browse PNG with similar name
            if browse_png is None:
                for candidate in sensor_dir.rglob("*.png"):
                    if base.replace("_d_img_", "_b_brw_") in candidate.stem or base.replace("_d_img_", "_b_brw_") == candidate.stem:
                        browse_png = candidate
                        break

            # Check geometry CSV
            if csv_path is None:
                geom_dir = sensor_dir / "geometry"
                if geom_dir.exists():
                    for candidate in geom_dir.rglob("*.csv"):
                        if base.replace("_d_img_", "_g_grd_") in candidate.stem:
                            csv_path = candidate
                            break

            size_mb = xml_path.stat().st_size / (1024 * 1024) if xml_path.exists() else 0
            records.append({
                "sensor": sensor,
                "product_type": product_type,
                "xml_path": str(xml_path),
                "img_path": str(img_path) if img_path else None,
                "hdr_path": str(hdr_path) if hdr_path else None,
                "qub_path": str(qub_path) if qub_path else None,
                "geometry_csv_path": str(csv_path) if csv_path else None,
                "browse_png_path": str(browse_png) if browse_png else None,
                "xml_size_mb": round(size_mb, 3),
            })

    df = pd.DataFrame(records)
    if not df.empty:
        # Add file sizes
        for col in ["img_path", "hdr_path", "qub_path", "geometry_csv_path", "browse_png_path"]:
            df[col + "_size_mb"] = df[col].apply(
                lambda x: round(Path(x).stat().st_size / (1024 * 1024), 3) if x and Path(x).exists() else 0
            )
    return df

discovered_df = discover_files()
if not discovered_df.empty:
    print("\n=== DISCOVERED FILES ===")
    display_cols = ["sensor", "product_type", "xml_path", "img_path", "hdr_path", "qub_path", "geometry_csv_path", "browse_png_path", "xml_size_mb"]
    print(discovered_df[display_cols].to_string())
    discovered_df.to_csv(RESULTS_DIR / "metadata" / "discovered_files.csv", index=False)
    logger.info(f"Discovered {len(discovered_df)} file records")
else:
    logger.warning("No files discovered in ROOT_DIR")

# Print package versions
print("\n=== PACKAGE VERSIONS ===")
for pkg_name in ["cv2", "numpy", "pandas", "matplotlib", "seaborn", "skimage", "rasterio", "spectral", "tifffile", "lxml"]:
    try:
        mod = __import__(pkg_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"{pkg_name}: {version}")
    except Exception as e:
        print(f"{pkg_name}: not available ({e})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing opencv-contrib-python-headless...
Installing scikit-image...
GPU available: Tesla T4

=== DISCOVERED FILES ===
  sensor product_type                                                                                                               xml_path                                                                                                           img_path                                                                                                    hdr_path                                                                                                    qub_path                                                                                                      geometry_csv_path                                                                                                      browse_png_path  xml_size_mb
0   iirs         data       

#  CELL GROUP C — Configuration Cell

In [2]:
CONFIG = {
    "mode": "synthetic_validation",  # valid modes: "real_pair", "synthetic_validation"
    "source_sensor": "tmc2",
    "reference_sensor": "ohrc",
    # Other valid pair choices:
    # source_sensor="ohrc", reference_sensor="tmc2"
    # source_sensor="tmc2", reference_sensor="iirs"
    # source_sensor="iirs", reference_sensor="tmc2"
    "use_browse_preview_first": True,
    "use_raw_data_if_readable": True,
    "max_working_dimension": 1600,
    "tile_size": 1024,
    "tile_overlap": 128,
    "n_grid_rows": 6,
    "n_grid_cols": 6,
    "max_matches_per_cell": 8,
    "ransac_reproj_threshold": 3.0,
    "min_inliers": 12,
    "min_inlier_ratio": 0.20,
    "min_coverage": 0.25,
    "accept_rcs": 75,
    "review_rcs": 50,
    "use_clahe": True,
    "use_homomorphic_filter": True,
    "use_phase_gradient_structural_view": True,
    "enable_orb": True,
    "enable_sift": True,
    "enable_akaze": True,
    "enable_optional_learned_matcher": False,
    "save_debug_images": True,
    "manual_roi_source": None,
    "manual_roi_reference": None,
    "synthetic_source_sensor": "ohrc",
    "synthetic_transform": {
        "rotation_deg": 7.0,
        "scale": 0.92,
        "tx_px": 45,
        "ty_px": -35,
        "perspective_jitter_px": 12,
    },
}

# Save config
with open(RESULTS_DIR / "metadata" / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

logger.info(f"CONFIG loaded: mode={CONFIG['mode']}, source={CONFIG['source_sensor']}, reference={CONFIG['reference_sensor']}")
print(f"CONFIG: mode={CONFIG['mode']}, source_sensor={CONFIG['source_sensor']}, reference_sensor={CONFIG['reference_sensor']}")
print("To change source/reference pair, edit CONFIG['source_sensor'] and CONFIG['reference_sensor'].")
print("To run synthetic validation, set CONFIG['mode'] = 'synthetic_validation'.")

CONFIG: mode=synthetic_validation, source_sensor=tmc2, reference_sensor=ohrc
To change source/reference pair, edit CONFIG['source_sensor'] and CONFIG['reference_sensor'].
To run synthetic validation, set CONFIG['mode'] = 'synthetic_validation'.


# CELL GROUP D — Metadata and File Reader Utilities

In [3]:
def parse_pds4_xml(xml_path: Union[str, Path]) -> Dict[str, Any]:
    """
    Parse a PDS4 XML file (Chandrayaan-2 ISRO format).
    Correctly reads <Axis_Array> blocks to extract lines/samples/bands.
    Handles namespaces transparently.
    """
    xml_path = Path(xml_path)
    metadata = {
        "xml_path": str(xml_path),
        "file_name": xml_path.name,
        "lines": None,
        "samples": None,
        "line_samples": None,
        "bands": None,
        "data_type": None,
        "byte_order": None,
        "offset": None,
        "scaling_factor": None,
        "value_offset": None,
        "valid_minimum": None,
        "valid_maximum": None,
        "missing_constant": None,
        "observation_time": None,
        "product_id": None,
        "instrument_id": None,
        "pixel_resolution_m": None,
        "spacecraft_altitude_km": None,
        "sun_azimuth_deg": None,
        "sun_elevation_deg": None,
        "corner_coords": {},
    }
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        def strip_ns(tag):
            return tag.split("}")[-1] if "}" in tag else tag

        def local_name(elem):
            return strip_ns(elem.tag).lower()

        # ---------- 1) Find the Array_2D_Image / Array_3D_Image block ----------
        array_elem = None
        for elem in root.iter():
            if local_name(elem) in ("array_2d_image", "array_2d", "array_3d_image", "array_3d"):
                array_elem = elem
                break

        lines = samples = bands = None
        data_type = None
        offset = None
        byte_order = None

        if array_elem is not None:
            # Get children by local name
            for child in array_elem:
                cname = local_name(child)
                ctext = child.text.strip() if child.text and child.text.strip() else None

                if cname == "element_array":
                    # Inside: data_type, unit
                    for sub in child:
                        sname = local_name(sub)
                        stext = sub.text.strip() if sub.text and sub.text.strip() else None
                        if sname == "data_type" and stext:
                            data_type = stext
                        elif sname == "unit" and stext and "byte" in stext.lower():
                            # Not the dtype, but let's see
                            pass

                elif cname == "axis_array":
                    # Contains: axis_name, elements, sequence_number
                    axis_name = None
                    axis_elements = None
                    for sub in child:
                        sname = local_name(sub)
                        stext = sub.text.strip() if sub.text and sub.text.strip() else None
                        if sname == "axis_name" and stext:
                            axis_name = stext.lower()
                        elif sname == "elements" and stext:
                            try:
                                axis_elements = int(stext)
                            except ValueError:
                                pass
                    if axis_name and axis_elements:
                        if "line" in axis_name or "row" in axis_name:
                            lines = axis_elements
                        elif "sample" in axis_name or "column" in axis_name or "pixel" in axis_name:
                            samples = axis_elements
                        elif "band" in axis_name:
                            bands = axis_elements

        # ---------- 2) Fallback: search for offset/data_type directly ----------
        # <offset> tag (data offset in bytes)
        for elem in root.iter():
            lname = local_name(elem)
            ltext = elem.text.strip() if elem.text and elem.text.strip() else None
            if not ltext:
                continue

            if lname == "offset" and offset is None:
                try:
                    offset = int(float(ltext))
                except ValueError:
                    pass
            elif lname == "data_type" and data_type is None:
                data_type = ltext
            elif lname == "byte_order" or lname == "bits_selection":
                if byte_order is None:
                    byte_order = ltext

        # ---------- 3) Other metadata ----------
        for elem in root.iter():
            lname = local_name(elem)
            ltext = elem.text.strip() if elem.text and elem.text.strip() else None
            if not ltext:
                continue

            if lname == "start_date_time" and metadata["observation_time"] is None:
                metadata["observation_time"] = ltext
            elif lname == "logical_identifier" and metadata["product_id"] is None:
                metadata["product_id"] = ltext
            elif lname == "name" and metadata["instrument_id"] is None:
                # First <name> inside Observing_System_Component is usually the instrument
                parent = elem.getparent() if hasattr(elem, "getparent") else None
                metadata["instrument_id"] = ltext  # crude; refined below
            elif lname == "pixel_resolution" and metadata["pixel_resolution_m"] is None:
                try:
                    metadata["pixel_resolution_m"] = float(ltext)
                except ValueError:
                    pass
            elif lname == "spacecraft_altitude" and metadata["spacecraft_altitude_km"] is None:
                try:
                    metadata["spacecraft_altitude_km"] = float(ltext)
                except ValueError:
                    pass
            elif lname == "sun_azimuth" and metadata["sun_azimuth_deg"] is None:
                try:
                    metadata["sun_azimuth_deg"] = float(ltext)
                except ValueError:
                    pass
            elif lname == "sun_elevation" and metadata["sun_elevation_deg"] is None:
                try:
                    metadata["sun_elevation_deg"] = float(ltext)
                except ValueError:
                    pass
            # Corner coordinates
            for c in ("upper_left_latitude", "upper_left_longitude",
                      "upper_right_latitude", "upper_right_longitude",
                      "lower_left_latitude", "lower_left_longitude",
                      "lower_right_latitude", "lower_right_longitude"):
                if lname == c and c not in metadata["corner_coords"]:
                    try:
                        metadata["corner_coords"][c] = float(ltext)
                    except ValueError:
                        pass

        # Assign final values
        metadata["lines"] = lines
        metadata["samples"] = samples
        metadata["line_samples"] = samples
        metadata["bands"] = bands if bands is not None else 1
        metadata["data_type"] = data_type
        metadata["offset"] = offset if offset is not None else 0
        metadata["byte_order"] = byte_order if byte_order is not None else "LSB"

        logger.info(f"Parsed {xml_path.name}: lines={lines}, samples={samples}, "
                    f"bands={metadata['bands']}, dtype={data_type}, offset={metadata['offset']}")

        if lines is None or samples is None:
            logger.warning(f"  Could not extract dimensions from {xml_path.name}. "
                           f"Array element found: {array_elem is not None}")

    except Exception as e:
        logger.error(f"Failed to parse PDS4 XML {xml_path}: {e}")
        metadata["error"] = str(e)
        metadata["traceback"] = traceback.format_exc()

    with open(RESULTS_DIR / "metadata" / f"{xml_path.stem}_pds4.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)
    return metadata


def parse_envi_header(hdr_path: Union[str, Path]) -> Dict[str, Any]:
    """Parse an ENVI header file."""
    hdr_path = Path(hdr_path)
    metadata = {
        "hdr_path": str(hdr_path), "file_name": hdr_path.name,
        "samples": None, "lines": None, "bands": None,
        "interleave": None, "data_type": None, "byte_order": None,
        "header_offset": None, "wavelengths": None, "ignore_value": None,
    }
    try:
        header = spectral.io.envi.read_envi_header(str(hdr_path))
        metadata["samples"] = int(header.get("samples", 0))
        metadata["lines"] = int(header.get("lines", 0))
        metadata["bands"] = int(header.get("bands", 0))
        metadata["interleave"] = header.get("interleave", "unknown")
        metadata["data_type"] = int(header.get("data type", 0))
        metadata["byte_order"] = int(header.get("byte order", 0))
        metadata["header_offset"] = int(header.get("header offset", 0))
        if "wavelength" in header:
            metadata["wavelengths"] = header["wavelength"]
        if "ignore value" in header:
            metadata["ignore_value"] = header["ignore value"]
        logger.info(f"Parsed ENVI header: {hdr_path.name} — "
                    f"{metadata['lines']}×{metadata['samples']}×{metadata['bands']}, "
                    f"interleave={metadata['interleave']}")
    except Exception as e:
        logger.error(f"Failed to parse ENVI header {hdr_path}: {e}")
        metadata["error"] = str(e)
        metadata["traceback"] = traceback.format_exc()

    with open(RESULTS_DIR / "metadata" / f"{hdr_path.stem}_envi.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)
    return metadata


def _pds4_dtype_map() -> Dict[str, np.dtype]:
    """PDS4 → numpy dtype mapping."""
    return {
        "UnsignedByte": np.dtype("u1"),
        "SignedByte": np.dtype("i1"),
        "UnsignedLSB2": np.dtype("<u2"),
        "SignedLSB2": np.dtype("<i2"),
        "UnsignedMSB2": np.dtype(">u2"),
        "SignedMSB2": np.dtype(">i2"),
        "UnsignedLSB4": np.dtype("<u4"),
        "SignedLSB4": np.dtype("<i4"),
        "UnsignedMSB4": np.dtype(">u4"),
        "SignedMSB4": np.dtype(">i4"),
        "IEEE754LSBSingle": np.dtype("<f4"),
        "IEEE754MSBSingle": np.dtype(">f4"),
        "IEEE754LSBDouble": np.dtype("<f8"),
        "IEEE754MSBDouble": np.dtype(">f8"),
        # Aliases
        "UnsignedByte": np.dtype("u1"),
        "Byte": np.dtype("u1"),
        "Unsigned Word": np.dtype("<u2"),
        "Unsigned Integer": np.dtype("<u4"),
        "IEEE754 Single": np.dtype("<f4"),
        "IEEE754 Double": np.dtype("<f8"),
    }


def inspect_binary_file(file_path: Union[str, Path], metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Inspect a binary file: compare expected vs actual size."""
    file_path = Path(file_path)
    result = {
        "file_path": str(file_path), "exists": file_path.exists(),
        "actual_size_bytes": None, "expected_size_bytes": None,
        "size_match": False, "warning": None,
    }
    if not file_path.exists():
        logger.error(f"Binary file does not exist: {file_path}")
        return result

    actual_size = file_path.stat().st_size
    result["actual_size_bytes"] = actual_size

    lines = metadata.get("lines")
    samples = metadata.get("samples")
    bands = metadata.get("bands", 1) or 1
    dtype_str = metadata.get("data_type", "")
    offset = metadata.get("offset", 0) or 0

    dtype_map = _pds4_dtype_map()
    if dtype_str in dtype_map and lines and samples:
        dt = dtype_map[dtype_str]
        expected = offset + lines * samples * bands * dt.itemsize
        result["expected_size_bytes"] = expected
        result["size_match"] = (expected == actual_size)
        if not result["size_match"]:
            logger.warning(f"Size mismatch for {file_path.name}: "
                           f"expected {expected}, actual {actual_size}")
    else:
        result["warning"] = "Metadata incomplete; cannot verify size."
    return result


def read_pds_img_with_fallback(
    img_path: Union[str, Path],
    xml_path: Union[str, Path],
    max_pixels: int = 100_000_000,
) -> Tuple[Optional[np.ndarray], Dict[str, Any]]:
    """
    Read a PDS4 IMG file. Reads the whole file or a strided subset.
    Returns (2D float32 array or None, metadata dict).
    """
    img_path = Path(img_path)
    xml_path = Path(xml_path)
    metadata = parse_pds4_xml(xml_path)
    result_info = {"method": None, "error": None, "traceback": None}

    # Try rasterio (unlikely to work on PDS4 but try)
    try:
        with rasterio.open(img_path) as src:
            data = src.read(1)
            if data.ndim == 2:
                arr = data.astype(np.float32)
                result_info["method"] = "rasterio"
                logger.info(f"Read {img_path.name} via rasterio, shape={arr.shape}")
                return arr, {**metadata, **result_info}
    except Exception as e:
        logger.warning(f"rasterio failed for {img_path.name}: {e}")

    # XML-assisted raw binary read
    try:
        lines = metadata.get("lines")
        samples = metadata.get("samples")
        bands = metadata.get("bands", 1) or 1
        dtype_str = metadata.get("data_type", "UnsignedByte")
        offset = int(metadata.get("offset", 0) or 0)
        byte_order_str = (metadata.get("byte_order") or "LSB").lower()

        if lines is None or samples is None:
            raise ValueError(
                f"Cannot read {img_path.name}: XML parser did not extract "
                f"lines/samples. Got lines={lines}, samples={samples}."
            )

        dtype_map = _pds4_dtype_map()
        if dtype_str not in dtype_map:
            # Try case-insensitive match
            found = None
            for k, v in dtype_map.items():
                if k.lower() == str(dtype_str).lower():
                    found = v
                    break
            if found is None:
                raise ValueError(f"Unsupported data type: {dtype_str}")
            dt = found
        else:
            dt = dtype_map[dtype_str]

        # Byte order fix
        if "msb" in byte_order_str or "big" in byte_order_str:
            if dt.itemsize > 1:
                dt = dt.newbyteorder(">")
        else:
            if dt.itemsize > 1:
                dt = dt.newbyteorder("<")

        itemsize = dt.itemsize
        n_pixels = lines * samples * bands

        # Size check
        file_size = img_path.stat().st_size
        expected_size = offset + n_pixels * itemsize
        logger.info(f"  Expected file size: {expected_size} bytes, "
                    f"actual: {file_size} bytes, "
                    f"match: {expected_size == file_size}")

        # If file doesn't match expected size, maybe offset is wrong
        if expected_size != file_size:
            # Try assuming offset=0
            if offset + n_pixels * itemsize != file_size and n_pixels * itemsize == file_size:
                logger.warning(f"  Offset {offset} seems wrong; using 0.")
                offset = 0
                expected_size = n_pixels * itemsize

        # Read
        with open(img_path, "rb") as f:
            f.seek(offset)
            raw = np.fromfile(f, dtype=dt, count=n_pixels)

        if raw.size != n_pixels:
            raise ValueError(f"Expected {n_pixels} elements, got {raw.size}")

        # Reshape: Chandrayaan-2 uses "Last Index Fastest" (C-order, samples fastest)
        # For 2D (bands=1): shape = (lines, samples)
        # For 3D: (bands, lines, samples) if BSQ-like, or (lines, samples, bands) if BIP
        # For Chandrayaan-2 OHRC/TMC-2, bands=1, so this is straightforward.
        if bands == 1:
            arr = raw.reshape((lines, samples)).astype(np.float32)
        else:
            arr = raw.reshape((bands, lines, samples)).astype(np.float32)
            # Use first band as representative
            arr = arr[0].astype(np.float32)

        # Sanitize
        arr[~np.isfinite(arr)] = np.nan

        result_info["method"] = "numpy_xml"
        logger.info(f"Read {img_path.name} via XML+numpy: shape={arr.shape}, "
                    f"dtype={dt}, offset={offset}")
        return arr, {**metadata, **result_info}

    except Exception as e:
        logger.error(f"Failed to read {img_path.name}: {e}")
        result_info["error"] = str(e)
        result_info["traceback"] = traceback.format_exc()
        with open(RESULTS_DIR / "logs" / f"read_error_{img_path.stem}.txt", "w") as f:
            f.write(traceback.format_exc())
        return None, {**metadata, **result_info}


def read_iirs_cube_with_fallback(
    qub_path: Union[str, Path],
    hdr_path: Union[str, Path],
    xml_path: Union[str, Path],
) -> Tuple[Optional[Any], Dict[str, Any]]:
    """Open IIRS cube lazily."""
    qub_path = Path(qub_path)
    hdr_path = Path(hdr_path)
    xml_path = Path(xml_path)
    envi_meta = parse_envi_header(hdr_path)
    pds_meta = parse_pds4_xml(xml_path)
    result_info = {"method": None, "error": None, "traceback": None}

    try:
        img = spectral.envi.open(str(hdr_path), str(qub_path))
        result_info["method"] = "spectral_envi"
        logger.info(f"Opened IIRS cube via spectral.envi: {img.shape}")
        return img, {**envi_meta, **pds_meta, **result_info}
    except Exception as e:
        logger.warning(f"spectral.envi.open failed: {e}")

    try:
        samples = envi_meta.get("samples")
        lines = envi_meta.get("lines")
        bands = envi_meta.get("bands")
        interleave = (envi_meta.get("interleave") or "bsq").lower()
        data_type = envi_meta.get("data_type", 4)
        header_offset = envi_meta.get("header_offset", 0)
        byte_order = envi_meta.get("byte_order", 0)

        dtype_map = {
            1: np.dtype("u1"), 2: np.dtype("i2"), 3: np.dtype("i4"),
            4: np.dtype("f4"), 5: np.dtype("f8"),
            12: np.dtype("u2"), 13: np.dtype("u4"), 14: np.dtype("i8"), 15: np.dtype("u8"),
        }
        if data_type not in dtype_map:
            raise ValueError(f"Unsupported ENVI data type: {data_type}")
        dt = dtype_map[data_type]
        dt = dt.newbyteorder(">" if byte_order == 1 else "<")

        if not (samples and lines and bands):
            raise ValueError("ENVI metadata incomplete")

        if interleave == "bsq":
            shape, order = (bands, lines, samples), "bsq"
        elif interleave == "bil":
            shape, order = (lines, bands, samples), "bil"
        elif interleave == "bip":
            shape, order = (lines, samples, bands), "bip"
        else:
            raise ValueError(f"Unsupported interleave: {interleave}")

        memmap = np.memmap(qub_path, dtype=dt, mode="r", offset=header_offset, shape=shape, order="C")
        result_info["method"] = "numpy_memmap"
        result_info["interleave"] = order
        logger.info(f"Opened IIRS cube via memmap: shape={shape}, interleave={order}")
        return memmap, {**envi_meta, **pds_meta, **result_info}
    except Exception as e:
        logger.error(f"Failed to read IIRS cube: {e}")
        result_info["error"] = str(e)
        result_info["traceback"] = traceback.format_exc()
        with open(RESULTS_DIR / "logs" / "read_error_iirs.txt", "w") as f:
            f.write(traceback.format_exc())
        return None, {**envi_meta, **pds_meta, **result_info}


def read_iirs_subset(cube, bands=None, rows=None, cols=None, max_spatial=512):
    """
    Read a spatial subset safely.
    For 3D cube, handles BSQ/BIL/BIP heuristically using axis sizes.
    """
    if cube is None:
        raise ValueError("Cube is None")

    # spectral.envi object
    if hasattr(cube, "read_subregion"):
        n_rows, n_cols, n_bands = cube.shape
        r0, r1 = (0, min(max_spatial, n_rows)) if rows is None else rows
        c0, c1 = (0, min(max_spatial, n_cols)) if cols is None else cols
        if bands is None:
            bands = list(range(min(10, n_bands)))
        return cube.read_subregion((r0, r1), (c0, c1), bands)

    # numpy memmap
    if cube.ndim == 3:
        n0, n1, n2 = cube.shape
        # Heuristic for interleave
        if n0 < n1 and n0 < n2:
            # BSQ: (bands, lines, samples)
            r0, r1 = (0, min(max_spatial, n1)) if rows is None else rows
            c0, c1 = (0, min(max_spatial, n2)) if cols is None else cols
            b = list(range(min(10, n0))) if bands is None else bands
            return cube[np.array(b), r0:r1, c0:c1]
        elif n2 < n0 and n2 < n1:
            # BIP: (lines, samples, bands)
            r0, r1 = (0, min(max_spatial, n0)) if rows is None else rows
            c0, c1 = (0, min(max_spatial, n1)) if cols is None else cols
            b = list(range(min(10, n2))) if bands is None else bands
            return cube[r0:r1, c0:c1, :][:, :, b]
        else:
            # BIL: (lines, bands, samples)
            r0, r1 = (0, min(max_spatial, n0)) if rows is None else rows
            c0, c1 = (0, min(max_spatial, n2)) if cols is None else cols
            b = list(range(min(10, n1))) if bands is None else bands
            return cube[r0:r1, np.array(b), c0:c1]
    else:
        r0, r1 = (0, min(max_spatial, cube.shape[0])) if rows is None else rows
        c0, c1 = (0, min(max_spatial, cube.shape[1])) if cols is None else cols
        return cube[r0:r1, c0:c1]


def load_geometry_csv(csv_path: Union[str, Path]) -> Dict[str, Any]:
    """Load a geometry CSV."""
    csv_path = Path(csv_path)
    result = {
        "csv_path": str(csv_path), "shape": None, "columns": None,
        "lat_min": None, "lat_max": None, "lon_min": None, "lon_max": None,
    }
    try:
        df = pd.read_csv(csv_path)
        result["shape"] = df.shape
        result["columns"] = list(df.columns)
        cols_lower = {c.lower(): c for c in df.columns}
        lat_col = next((cols_lower[c] for c in ["lat", "latitude"] if c in cols_lower), None)
        lon_col = next((cols_lower[c] for c in ["lon", "longitude", "long"] if c in cols_lower), None)
        if lat_col:
            result["lat_min"] = float(df[lat_col].min())
            result["lat_max"] = float(df[lat_col].max())
        if lon_col:
            result["lon_min"] = float(df[lon_col].min())
            result["lon_max"] = float(df[lon_col].max())
        logger.info(f"Geometry CSV {csv_path.name}: shape={df.shape}, "
                    f"lat=[{result['lat_min']}, {result['lat_max']}], "
                    f"lon=[{result['lon_min']}, {result['lon_max']}]")
    except Exception as e:
        logger.error(f"Failed to load geometry CSV: {e}")
        result["error"] = str(e)

    with open(RESULTS_DIR / "metadata" / f"{csv_path.stem}_geometry.json", "w") as f:
        json.dump(result, f, indent=2, default=str)
    return result


def normalize_to_uint8(image, lower_percentile=2, upper_percentile=98):
    """Normalize to uint8."""
    if image is None:
        return None
    img = image.astype(np.float32)
    valid = np.isfinite(img)
    if not np.any(valid):
        return np.zeros(img.shape, dtype=np.uint8)
    lo = np.percentile(img[valid], lower_percentile)
    hi = np.percentile(img[valid], upper_percentile)
    if hi <= lo:
        return np.zeros(img.shape, dtype=np.uint8)
    out = np.clip((img - lo) / (hi - lo), 0, 1)
    out[~valid] = 0
    return (out * 255).astype(np.uint8)


def save_preview(image, title, out_path, max_dim=1024):
    """Save a PNG preview (downscaled)."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if image is not None:
        h, w = image.shape[:2]
        if max(h, w) > max_dim:
            scale = max_dim / max(h, w)
            image = cv2.resize(image, (max(1, int(w*scale)), max(1, int(h*scale))),
                               interpolation=cv2.INTER_AREA)
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    if image is None:
        ax.text(0.5, 0.5, "No image", ha="center", va="center")
    elif image.ndim == 2:
        ax.imshow(image, cmap="gray")
    else:
        ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=80, bbox_inches="tight")
    plt.close(fig)

#CELL GROUP E — Load and Inspect Actual Senso

In [4]:
def load_sensor_data(sensor: str, use_browse_first: bool = True, use_raw: bool = True) -> Dict[str, Any]:
    """Load data for a given sensor (ohrc, tmc2, iirs)."""
    result = {
        "sensor": sensor, "browse_image": None, "browse_path": None,
        "raw_image": None, "raw_path": None, "metadata": None,
        "geometry": None, "error": None, "used_fallback": False,
    }

    sensor_dir = ROOT_DIR / sensor
    if not sensor_dir.exists():
        result["error"] = f"Sensor directory not found: {sensor_dir}"
        logger.error(result["error"])
        return result

    # Browse PNG
    browse_png = None
    for p in sensor_dir.rglob("*.png"):
        if "_b_brw_" in p.name or "_brw_" in p.name:
            browse_png = p
            break

    # Raw data
    raw_img = raw_hdr = raw_qub = raw_xml = geom_csv = None

    if sensor == "iirs":
        for p in sensor_dir.rglob("*.qub"):
            raw_qub = p
            break
        if raw_qub:
            raw_hdr = raw_qub.with_suffix(".hdr")
            raw_xml = raw_qub.with_suffix(".xml")
    else:
        for p in sensor_dir.rglob("*.img"):
            raw_img = p
            break
        if raw_img:
            raw_xml = raw_img.with_suffix(".xml")

    geom_dir = sensor_dir / "geometry"
    if geom_dir.exists():
        for p in geom_dir.rglob("*.csv"):
            geom_csv = p
            break

    # Load browse preview
    if browse_png and browse_png.exists():
        try:
            browse_img = cv2.imread(str(browse_png), cv2.IMREAD_UNCHANGED)
            if browse_img is not None:
                if browse_img.ndim == 3:
                    browse_img = cv2.cvtColor(browse_img, cv2.COLOR_BGR2GRAY)
                result["browse_image"] = browse_img
                result["browse_path"] = str(browse_png)
                logger.info(f"Loaded browse preview for {sensor}: shape={browse_img.shape}")
                save_preview(browse_img, f"{sensor} Browse Preview",
                             RESULTS_DIR / "previews" / f"{sensor}_browse.png")
        except Exception as e:
            logger.error(f"Failed to load browse PNG for {sensor}: {e}")

    # Load raw data
    if use_raw:
        if sensor == "iirs" and raw_qub and raw_hdr and raw_xml:
            cube, meta = read_iirs_cube_with_fallback(raw_qub, raw_hdr, raw_xml)
            if cube is not None:
                result["metadata"] = meta
                try:
                    subset = read_iirs_subset(cube)
                    if subset.ndim == 3:
                        band_img = subset[0].astype(np.float32)
                    else:
                        band_img = subset.astype(np.float32)
                    band_img = normalize_to_uint8(band_img)
                    result["raw_image"] = band_img
                    result["raw_path"] = str(raw_qub)
                    save_preview(band_img, f"{sensor} Selected Band",
                                 RESULTS_DIR / "previews" / f"{sensor}_selected_band.png")
                    logger.info(f"Loaded IIRS subset: shape={subset.shape}")
                except Exception as e:
                    logger.error(f"Failed to create IIRS structural preview: {e}")
                    result["used_fallback"] = True
            else:
                result["used_fallback"] = True
                logger.warning(f"RAW PRODUCT PARSING FAILED FOR IIRS — USING BROWSE PREVIEW ONLY FOR DEMO.")

        elif sensor in ("ohrc", "tmc2") and raw_img and raw_xml:
            raw_arr, meta = read_pds_img_with_fallback(raw_img, raw_xml)
            if raw_arr is not None:
                # Downscale raw before storing to avoid RAM blow-ups
                h, w = raw_arr.shape
                if h * w > 20_000_000:  # > 20 M pixels
                    scale = np.sqrt(20_000_000 / (h * w))
                    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
                    logger.warning(f"Downsampling raw {sensor} from {raw_arr.shape} to ({new_h}, {new_w}) to save RAM")
                    raw_arr = cv2.resize(raw_arr, (new_w, new_h), interpolation=cv2.INTER_AREA)
                result["raw_image"] = normalize_to_uint8(raw_arr)
                result["raw_path"] = str(raw_img)
                result["metadata"] = meta
                save_preview(result["raw_image"], f"{sensor} Raw Calibrated",
                             RESULTS_DIR / "previews" / f"{sensor}_raw.png")
                logger.info(f"Loaded raw {sensor}: shape={result['raw_image'].shape}")
            else:
                result["used_fallback"] = True
                logger.warning(f"RAW PRODUCT PARSING FAILED FOR {sensor.upper()} — USING BROWSE PREVIEW ONLY FOR DEMO.")
        else:
            result["used_fallback"] = True
            logger.warning(f"No raw data found for {sensor}. Using browse preview only.")

    if geom_csv and geom_csv.exists():
        result["geometry"] = load_geometry_csv(geom_csv)

    if result["raw_image"] is None and result["browse_image"] is not None:
        result["raw_image"] = result["browse_image"]
        result["raw_path"] = result["browse_path"]
        result["used_fallback"] = True
        logger.info(f"Using browse preview as raw fallback for {sensor}")

    return result


# Load all sensors
print("\n=== LOADING SENSOR DATA ===")
sensor_data = {}
for sensor in ["ohrc", "tmc2", "iirs"]:
    print(f"\n--- Loading {sensor.upper()} ---")
    data = load_sensor_data(sensor, use_browse_first=CONFIG["use_browse_preview_first"], use_raw=CONFIG["use_raw_data_if_readable"])
    sensor_data[sensor] = data
    if data["raw_image"] is not None:
        print(f"  {sensor.upper()}: image shape={data['raw_image'].shape}, dtype={data['raw_image'].dtype}, range=[{data['raw_image'].min()}, {data['raw_image'].max()}]")
    else:
        print(f"  {sensor.upper()}: no image loaded. Error: {data.get('error', 'unknown')}")
    if data["metadata"]:
        print(f"  Metadata keys: {list(data['metadata'].keys())[:10]}...")
    if data["geometry"]:
        print(f"  Geometry: shape={data['geometry']['shape']}, lat=[{data['geometry']['lat_min']}, {data['geometry']['lat_max']}], lon=[{data['geometry']['lon_min']}, {data['geometry']['lon_max']}]")

# Save load summary
load_summary = []
for sensor, data in sensor_data.items():
    load_summary.append({
        "sensor": sensor,
        "browse_loaded": data["browse_image"] is not None,
        "raw_loaded": data["raw_image"] is not None,
        "used_fallback": data["used_fallback"],
        "error": data["error"],
        "image_shape": str(data["raw_image"].shape) if data["raw_image"] is not None else None,
    })
pd.DataFrame(load_summary).to_csv(RESULTS_DIR / "metadata" / "load_summary.csv", index=False)


=== LOADING SENSOR DATA ===

--- Loading OHRC ---


  OHRC: image shape=(11300, 1769), dtype=uint8, range=[0, 255]
  Metadata keys: ['xml_path', 'file_name', 'lines', 'samples', 'line_samples', 'bands', 'data_type', 'byte_order', 'offset', 'scaling_factor']...
  Geometry: shape=(92928, 4), lat=[4.7407567, 5.5568659], lon=[234.396774, 234.528638]

--- Loading TMC2 ---


  TMC2: image shape=(29499, 677), dtype=uint8, range=[0, 255]
  Metadata keys: ['xml_path', 'file_name', 'lines', 'samples', 'line_samples', 'bands', 'data_type', 'byte_order', 'offset', 'scaling_factor']...
  Geometry: shape=(71422, 4), lat=[-0.368352408, 29.0203984], lon=[234.123049, 235.051837]

--- Loading IIRS ---


  IIRS: image shape=(250, 10), dtype=uint8, range=[0, 255]
  Metadata keys: ['hdr_path', 'file_name', 'samples', 'lines', 'bands', 'interleave', 'data_type', 'byte_order', 'header_offset', 'wavelengths']...


# CELL GROUP F — Structural Preprocessing

In [5]:
def create_structural_view(image: np.ndarray, sensor_name: str, config: dict) -> Dict[str, np.ndarray]:
    """
    Create structural representations for matching.
    Returns a dictionary with normalized grayscale, CLAHE, homomorphic, gradient, structure, shadow mask, and final structural view.
    """
    result = {}
    if image is None:
        logger.error(f"Cannot create structural view for {sensor_name}: image is None")
        return result

    # Ensure grayscale
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    img = image.astype(np.float32)
    valid = np.isfinite(img)
    if not np.any(valid):
        logger.error(f"Image for {sensor_name} has no valid pixels")
        return result

    # Normalize to 0-1
    lo, hi = np.percentile(img[valid], [2, 98])
    if hi <= lo:
        img_norm = np.zeros_like(img)
    else:
        img_norm = np.clip((img - lo) / (hi - lo), 0, 1)
    img_norm[~valid] = 0
    result["normalized"] = img_norm

    # CLAHE
    if config.get("use_clahe", True):
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_uint8 = (img_norm * 255).astype(np.uint8)
        clahe_img = clahe.apply(img_uint8)
        result["clahe"] = clahe_img.astype(np.float32) / 255.0
    else:
        result["clahe"] = img_norm.copy()

    # Homomorphic filter
    if config.get("use_homomorphic_filter", True):
        # Log transform
        log_img = np.log1p(img_norm * 255.0)
        # Gaussian low-pass for illumination estimate
        sigma = 30
        illum = cv2.GaussianBlur(log_img, (0, 0), sigma)
        # Subtract illumination
        homomorphic = log_img - illum
        # Normalize
        homomorphic = normalize_to_uint8(homomorphic) / 255.0
        result["homomorphic"] = homomorphic
    else:
        result["homomorphic"] = img_norm.copy()

    # Gradient magnitude
    grad_x = cv2.Sobel(result["clahe"], cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(result["clahe"], cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    grad_mag = normalize_to_uint8(grad_mag) / 255.0
    result["gradient"] = grad_mag

    # Phase/edge structure
    if config.get("use_phase_gradient_structural_view", True):
        edges = cv2.Canny((result["clahe"] * 255).astype(np.uint8), 50, 150)
        # Combine edges with gradient
        structure = np.maximum(edges.astype(np.float32) / 255.0, grad_mag * 0.5)
        structure = cv2.GaussianBlur(structure, (3, 3), 0)
        result["structure"] = structure
    else:
        result["structure"] = grad_mag.copy()

    # Shadow/low-confidence mask
    # Low intensity percentile plus local contrast
    low_intensity = np.percentile(img_norm[valid], 10)
    shadow_mask = (img_norm <= low_intensity).astype(np.uint8)
    # Local contrast
    local_mean = cv2.blur(img_norm, (15, 15))
    local_std = cv2.blur((img_norm - local_mean)**2, (15, 15))
    local_std = np.sqrt(np.maximum(local_std, 0))
    low_contrast = (local_std < np.percentile(local_std[valid], 10)).astype(np.uint8)
    shadow_mask = np.maximum(shadow_mask, low_contrast)
    # Dilate
    shadow_mask = cv2.dilate(shadow_mask, np.ones((5, 5), np.uint8), iterations=1)
    result["shadow_mask"] = shadow_mask

    # Final structural view for matching
    final = result["structure"].copy()
    final[shadow_mask > 0] = 0
    result["final"] = final

    # Save panel
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(result["normalized"], cmap="gray")
    axes[0, 0].set_title("Normalized")
    axes[0, 1].imshow(result["clahe"], cmap="gray")
    axes[0, 1].set_title("CLAHE")
    axes[0, 2].imshow(result["homomorphic"], cmap="gray")
    axes[0, 2].set_title("Homomorphic")
    axes[1, 0].imshow(result["gradient"], cmap="gray")
    axes[1, 0].set_title("Gradient")
    axes[1, 1].imshow(result["structure"], cmap="gray")
    axes[1, 1].set_title("Structure")
    axes[1, 2].imshow(result["shadow_mask"], cmap="gray")
    axes[1, 2].set_title("Shadow Mask")
    for ax in axes.flat:
        ax.axis("off")
    plt.suptitle(f"{sensor_name} Structural Preprocessing")
    plt.tight_layout()
    panel_path = RESULTS_DIR / "structural_views" / f"{sensor_name}_structural_panel.png"
    plt.savefig(panel_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    logger.info(f"Saved structural panel: {panel_path}")

    return result


# Create structural views for loaded sensors
print("\n=== CREATING STRUCTURAL VIEWS ===")
structural_views = {}
for sensor, data in sensor_data.items():
    if data["raw_image"] is not None:
        print(f"Processing {sensor.upper()}...")
        sv = create_structural_view(data["raw_image"], sensor, CONFIG)
        structural_views[sensor] = sv
        if sv:
            print(f"  Structural view created. Final shape: {sv['final'].shape}")
    else:
        print(f"Skipping {sensor.upper()}: no image loaded.")


=== CREATING STRUCTURAL VIEWS ===
Processing OHRC...
  Structural view created. Final shape: (11300, 1769)
Processing TMC2...
  Structural view created. Final shape: (29499, 677)
Processing IIRS...
  Structural view created. Final shape: (250, 10)


# CELL GROUP G — Resolution and Pyramid Handling

In [6]:
def get_working_image(image: np.ndarray, max_dim: int) -> Tuple[np.ndarray, float, float]:
    """
    Resize image to fit within max_dim, preserving aspect ratio.
    Returns (resized_image, scale_x, scale_y).
    """
    if image is None:
        return None, 1.0, 1.0
    h, w = image.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)
    if scale >= 1.0:
        return image.copy(), 1.0, 1.0
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    scale_x = w / new_w
    scale_y = h / new_h
    return resized, scale_x, scale_y


def gaussian_pyramid(image: np.ndarray, levels: int = 3) -> List[np.ndarray]:
    """
    Build a Gaussian pyramid.
    """
    pyramid = [image]
    current = image
    for _ in range(levels - 1):
        current = cv2.pyrDown(current)
        pyramid.append(current)
    return pyramid


def choose_working_resolution(source: np.ndarray, reference: np.ndarray, max_dim: int) -> Dict[str, Any]:
    """
    Choose compatible working resolution for source and reference.
    Returns a dictionary with working images and scale factors.
    """
    result = {}
    src_working, src_sx, src_sy = get_working_image(source, max_dim)
    ref_working, ref_sx, ref_sy = get_working_image(reference, max_dim)
    result["source_working"] = src_working
    result["reference_working"] = ref_working
    result["source_scale_x"] = src_sx
    result["source_scale_y"] = src_sy
    result["reference_scale_x"] = ref_sx
    result["reference_scale_y"] = ref_sy
    result["source_original_shape"] = source.shape[:2]
    result["reference_original_shape"] = reference.shape[:2]
    result["source_working_shape"] = src_working.shape[:2] if src_working is not None else None
    result["reference_working_shape"] = ref_working.shape[:2] if ref_working is not None else None
    logger.info(f"Working resolution: source {result['source_original_shape']} -> {result['source_working_shape']}, "
                f"reference {result['reference_original_shape']} -> {result['reference_working_shape']}")
    return result


def crop_roi(image: np.ndarray, roi: Optional[List[int]]) -> np.ndarray:
    """
    Crop a manual ROI [x, y, width, height].
    """
    if image is None or roi is None:
        return image
    x, y, w, h = roi
    h_img, w_img = image.shape[:2]
    x = max(0, min(x, w_img - 1))
    y = max(0, min(y, h_img - 1))
    w = min(w, w_img - x)
    h = min(h, h_img - y)
    cropped = image[y:y+h, x:x+w]
    logger.info(f"Cropped ROI: x={x}, y={y}, w={w}, h={h}")
    return cropped

# CELL GROUP H — Adaptive Classical Matcher Cascade

In [7]:
def match_sift(source: np.ndarray, reference: np.ndarray, mask_source: Optional[np.ndarray] = None, mask_reference: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """
    SIFT + BFMatcher KNN + Lowe ratio test with mutual nearest-neighbour.
    """
    result = {
        "name": "SIFT",
        "keypoints_source": np.array([]),
        "keypoints_reference": np.array([]),
        "raw_match_count": 0,
        "match_distances": np.array([]),
        "feature_confidence": None,
        "debug_info": {},
    }
    if not hasattr(cv2, "SIFT_create"):
        result["debug_info"]["error"] = "SIFT not available"
        logger.warning("SIFT not available")
        return result

    try:
        if hasattr(cv2, "SIFT_create"):
              sift = cv2.SIFT_create(nfeatures=5000)
        elif hasattr(cv2, "SIFT"):
              sift = cv2.SIFT.create(nfeatures=5000)
        else:
              result["debug_info"]["error"] = "SIFT not available"
              logger.warning("SIFT not available in this OpenCV build")
              return result
        kp1, des1 = sift.detectAndCompute((source * 255).astype(np.uint8), mask_source)
        kp2, des2 = sift.detectAndCompute((reference * 255).astype(np.uint8), mask_reference)
        if des1 is None or des2 is None or len(kp1) < 4 or len(kp2) < 4:
            result["debug_info"]["error"] = "Insufficient keypoints"
            return result

        bf = cv2.BFMatcher(cv2.NORM_L2)
        matches = bf.knnMatch(des1, des2, k=2)
        # Lowe ratio test
        good = []
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                good.append(m)

        # Mutual nearest-neighbour
        matches_rev = bf.knnMatch(des2, des1, k=2)
        good_rev = []
        for m, n in matches_rev:
            if m.distance < 0.75 * n.distance:
                good_rev.append(m)

        # Build mutual set
        mutual = []
        rev_set = set()
        for m in good_rev:
            rev_set.add((m.queryIdx, m.trainIdx))
        for m in good:
            if (m.trainIdx, m.queryIdx) in rev_set:
                mutual.append(m)

        if len(mutual) < 4:
            result["debug_info"]["error"] = f"Only {len(mutual)} mutual matches"
            return result

        pts1 = np.float32([kp1[m.queryIdx].pt for m in mutual])
        pts2 = np.float32([kp2[m.trainIdx].pt for m in mutual])
        distances = np.array([m.distance for m in mutual])

        result["keypoints_source"] = pts1
        result["keypoints_reference"] = pts2
        result["raw_match_count"] = len(mutual)
        result["match_distances"] = distances
        result["feature_confidence"] = 1.0 - (distances / (distances.max() + 1e-6))
        result["debug_info"]["n_keypoints_source"] = len(kp1)
        result["debug_info"]["n_keypoints_reference"] = len(kp2)
        result["debug_info"]["n_good"] = len(good)
        result["debug_info"]["n_mutual"] = len(mutual)
        logger.info(f"SIFT: {len(kp1)} src kp, {len(kp2)} ref kp, {len(good)} good, {len(mutual)} mutual")
    except Exception as e:
        result["debug_info"]["error"] = str(e)
        logger.error(f"SIFT failed: {e}")
    return result


def match_orb(source: np.ndarray, reference: np.ndarray, mask_source: Optional[np.ndarray] = None, mask_reference: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """
    ORB + BFMatcher Hamming + KNN ratio test with mutual nearest-neighbour.
    """
    result = {
        "name": "ORB",
        "keypoints_source": np.array([]),
        "keypoints_reference": np.array([]),
        "raw_match_count": 0,
        "match_distances": np.array([]),
        "feature_confidence": None,
        "debug_info": {},
    }
    try:
        if hasattr(cv2, "ORB_create"):
            orb = cv2.ORB_create(nfeatures=5000)
        elif hasattr(cv2, "ORB"):
            orb = cv2.ORB.create(nfeatures=5000)
        else:
            result["debug_info"]["error"] = "ORB not available"
            return result
        kp1, des1 = orb.detectAndCompute((source * 255).astype(np.uint8), mask_source)
        kp2, des2 = orb.detectAndCompute((reference * 255).astype(np.uint8), mask_reference)
        if des1 is None or des2 is None or len(kp1) < 4 or len(kp2) < 4:
            result["debug_info"]["error"] = "Insufficient keypoints"
            return result

        bf = cv2.BFMatcher(cv2.NORM_HAMMING)
        matches = bf.knnMatch(des1, des2, k=2)
        good = []
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                good.append(m)

        matches_rev = bf.knnMatch(des2, des1, k=2)
        good_rev = []
        for m, n in matches_rev:
            if m.distance < 0.75 * n.distance:
                good_rev.append(m)

        mutual = []
        rev_set = set()
        for m in good_rev:
            rev_set.add((m.queryIdx, m.trainIdx))
        for m in good:
            if (m.trainIdx, m.queryIdx) in rev_set:
                mutual.append(m)

        if len(mutual) < 4:
            result["debug_info"]["error"] = f"Only {len(mutual)} mutual matches"
            return result

        pts1 = np.float32([kp1[m.queryIdx].pt for m in mutual])
        pts2 = np.float32([kp2[m.trainIdx].pt for m in mutual])
        distances = np.array([m.distance for m in mutual])

        result["keypoints_source"] = pts1
        result["keypoints_reference"] = pts2
        result["raw_match_count"] = len(mutual)
        result["match_distances"] = distances
        result["feature_confidence"] = 1.0 - (distances / (distances.max() + 1e-6))
        result["debug_info"]["n_keypoints_source"] = len(kp1)
        result["debug_info"]["n_keypoints_reference"] = len(kp2)
        result["debug_info"]["n_good"] = len(good)
        result["debug_info"]["n_mutual"] = len(mutual)
        logger.info(f"ORB: {len(kp1)} src kp, {len(kp2)} ref kp, {len(good)} good, {len(mutual)} mutual")
    except Exception as e:
        result["debug_info"]["error"] = str(e)
        logger.error(f"ORB failed: {e}")
    return result


def match_akaze(source: np.ndarray, reference: np.ndarray, mask_source: Optional[np.ndarray] = None, mask_reference: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """
    AKAZE + BFMatcher Hamming + KNN ratio test with mutual nearest-neighbour.
    """
    result = {
        "name": "AKAZE",
        "keypoints_source": np.array([]),
        "keypoints_reference": np.array([]),
        "raw_match_count": 0,
        "match_distances": np.array([]),
        "feature_confidence": None,
        "debug_info": {},
    }
    try:
        if hasattr(cv2, "AKAZE_create"):
            akaze = cv2.AKAZE_create()
        elif hasattr(cv2, "AKAZE"):
            akaze = cv2.AKAZE.create()
        else:
            result["debug_info"]["error"] = "AKAZE not available"
            logger.warning("AKAZE not available in this OpenCV build")
            return result
        kp1, des1 = akaze.detectAndCompute((source * 255).astype(np.uint8), mask_source)
        kp2, des2 = akaze.detectAndCompute((reference * 255).astype(np.uint8), mask_reference)
        if des1 is None or des2 is None or len(kp1) < 4 or len(kp2) < 4:
            result["debug_info"]["error"] = "Insufficient keypoints"
            return result

        bf = cv2.BFMatcher(cv2.NORM_HAMMING)
        matches = bf.knnMatch(des1, des2, k=2)
        good = []
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                good.append(m)

        matches_rev = bf.knnMatch(des2, des1, k=2)
        good_rev = []
        for m, n in matches_rev:
            if m.distance < 0.75 * n.distance:
                good_rev.append(m)

        mutual = []
        rev_set = set()
        for m in good_rev:
            rev_set.add((m.queryIdx, m.trainIdx))
        for m in good:
            if (m.trainIdx, m.queryIdx) in rev_set:
                mutual.append(m)

        if len(mutual) < 4:
            result["debug_info"]["error"] = f"Only {len(mutual)} mutual matches"
            return result

        pts1 = np.float32([kp1[m.queryIdx].pt for m in mutual])
        pts2 = np.float32([kp2[m.trainIdx].pt for m in mutual])
        distances = np.array([m.distance for m in mutual])

        result["keypoints_source"] = pts1
        result["keypoints_reference"] = pts2
        result["raw_match_count"] = len(mutual)
        result["match_distances"] = distances
        result["feature_confidence"] = 1.0 - (distances / (distances.max() + 1e-6))
        result["debug_info"]["n_keypoints_source"] = len(kp1)
        result["debug_info"]["n_keypoints_reference"] = len(kp2)
        result["debug_info"]["n_good"] = len(good)
        result["debug_info"]["n_mutual"] = len(mutual)
        logger.info(f"AKAZE: {len(kp1)} src kp, {len(kp2)} ref kp, {len(good)} good, {len(mutual)} mutual")
    except Exception as e:
        result["debug_info"]["error"] = str(e)
        logger.error(f"AKAZE failed: {e}")
    return result


def run_adaptive_matcher_cascade(source: np.ndarray, reference: np.ndarray,
                                  mask_source: Optional[np.ndarray] = None,
                                  mask_reference: Optional[np.ndarray] = None,
                                  config: dict = None) -> List[Dict[str, Any]]:
    """
    Run the adaptive matcher cascade: SIFT first, then ORB/AKAZE if SIFT fails or has poor provisional quality.
    Returns a list of matcher results.
    """
    results = []
    if config is None:
        config = CONFIG

    # Run SIFT first
    if config.get("enable_sift", True) and hasattr(cv2, "SIFT_create"):
        print("Running SIFT matcher...")
        sift_result = match_sift(source, reference, mask_source, mask_reference)
        results.append(sift_result)
        if sift_result["raw_match_count"] >= 20:
            logger.info("SIFT has sufficient tentative matches. Proceeding with verification.")
            return results
        else:
            logger.info(f"SIFT has only {sift_result['raw_match_count']} matches. Trying ORB/AKAZE.")

    # Run ORB
    if config.get("enable_orb", True):
        print("Running ORB matcher...")
        orb_result = match_orb(source, reference, mask_source, mask_reference)
        results.append(orb_result)

    if config.get("enable_akaze", True) and (hasattr(cv2, "AKAZE_create") or hasattr(cv2, "AKAZE")):
        print("Running AKAZE matcher...")
        results.append(match_akaze(source, reference, mask_source, mask_reference))
    elif config.get("enable_akaze", True):
        logger.info("AKAZE not available; skipping")

    # Run AKAZE
    if config.get("enable_akaze", True):
        print("Running AKAZE matcher...")
        akaze_result = match_akaze(source, reference, mask_source, mask_reference)
        results.append(akaze_result)

    # Optional learned matcher
    if config.get("enable_optional_learned_matcher", False):
        logger.info("Optional learned matcher enabled but not implemented in this MVP. Skipping.")
        print("Optional learned matcher unavailable. Using classical matchers only.")

    return results

# CELL GROUP I — Geometry Verification with MAGSAC/USAC

In [8]:
def fit_transform_and_verify(src_pts: np.ndarray, ref_pts: np.ndarray, config: dict) -> Dict[str, Any]:
    """
    Fit homography/affine and verify geometry using MAGSAC/USAC.
    Returns a dictionary with transform matrix, inlier mask, method, and diagnostics.
    """
    result = {
        "success": False,
        "H": None,
        "affine": None,
        "inlier_mask": None,
        "method": None,
        "model_type": None,
        "diagnostics": {},
    }
    if src_pts is None or ref_pts is None or len(src_pts) < 4:
        result["diagnostics"]["error"] = "Insufficient points (need >= 4)"
        return result

    src_pts = np.float32(src_pts)
    ref_pts = np.float32(ref_pts)

    # Try homography with USAC_MAGSAC
    H = None
    inlier_mask_h = None
    method_h = None

    try:
        if hasattr(cv2, "USAC_MAGSAC"):
            H, inlier_mask_h = cv2.findHomography(
                src_pts, ref_pts, cv2.USAC_MAGSAC,
                ransacReprojThreshold=config["ransac_reproj_threshold"],
                maxIters=10000, confidence=0.999
            )
            method_h = "USAC_MAGSAC"
        else:
            H, inlier_mask_h = cv2.findHomography(
                src_pts, ref_pts, cv2.RANSAC,
                ransacReprojThreshold=config["ransac_reproj_threshold"],
                maxIters=10000, confidence=0.999
            )
            method_h = "RANSAC"
    except Exception as e:
        logger.warning(f"Homography fitting failed: {e}")

    # Try affine with USAC/MAGSAC
    affine = None
    inlier_mask_a = None
    method_a = None
    try:
        if hasattr(cv2, "USAC_MAGSAC"):
            affine, inlier_mask_a = cv2.estimateAffinePartial2D(
                src_pts, ref_pts, method=cv2.USAC_MAGSAC,
                ransacReprojThreshold=config["ransac_reproj_threshold"],
                maxIters=10000, confidence=0.999
            )
            method_a = "USAC_MAGSAC_affine"
        else:
            affine, inlier_mask_a = cv2.estimateAffinePartial2D(
                src_pts, ref_pts, method=cv2.RANSAC,
                ransacReprojThreshold=config["ransac_reproj_threshold"],
                maxIters=10000, confidence=0.999
            )
            method_a = "RANSAC_affine"
    except Exception as e:
        logger.warning(f"Affine fitting failed: {e}")

    # Choose best model
    best = None
    if H is not None and inlier_mask_h is not None:
        n_inliers_h = int(np.sum(inlier_mask_h))
        if n_inliers_h >= 4:
            # Compute RMSE for H
            src_h = cv2.convertPointsToHomogeneous(src_pts).reshape(-1, 3).T
            proj = H @ src_h
            proj = proj[:2] / (proj[2] + 1e-12)
            proj = proj.T
            errors = np.linalg.norm(proj - ref_pts, axis=1)
            rmse_h = np.sqrt(np.mean(errors[inlier_mask_h.ravel() == 1]**2)) if n_inliers_h > 0 else np.inf
            best = {
                "model": "homography", "H": H, "inlier_mask": inlier_mask_h.ravel(),
                "method": method_h, "n_inliers": n_inliers_h, "rmse": rmse_h,
            }

    if affine is not None and inlier_mask_a is not None:
        n_inliers_a = int(np.sum(inlier_mask_a))
        if n_inliers_a >= 4:
            # Compute RMSE for affine
            src_a = np.hstack([src_pts, np.ones((len(src_pts), 1))])
            proj = (affine @ src_a.T).T
            errors = np.linalg.norm(proj - ref_pts, axis=1)
            rmse_a = np.sqrt(np.mean(errors[inlier_mask_a.ravel() == 1]**2)) if n_inliers_a > 0 else np.inf
            if best is None or (n_inliers_a > best["n_inliers"]) or (n_inliers_a == best["n_inliers"] and rmse_a < best["rmse"]):
                best = {
                    "model": "affine", "affine": affine, "inlier_mask": inlier_mask_a.ravel(),
                    "method": method_a, "n_inliers": n_inliers_a, "rmse": rmse_a,
                }

    if best is None:
        result["diagnostics"]["error"] = "No valid transform found"
        return result

    # Validate transform
    if best["model"] == "homography":
        H_mat = best["H"]
        if not np.all(np.isfinite(H_mat)):
            result["diagnostics"]["error"] = "Homography contains non-finite values"
            return result
        det = np.linalg.det(H_mat)
        if abs(det) < 1e-12:
            result["diagnostics"]["error"] = f"Homography is singular (det={det})"
            return result
        cond = np.linalg.cond(H_mat)
        best["determinant"] = det
        best["condition_number"] = cond
    else:
        affine_mat = best["affine"]
        if not np.all(np.isfinite(affine_mat)):
            result["diagnostics"]["error"] = "Affine contains non-finite values"
            return result
        det = np.linalg.det(affine_mat[:, :2])
        if abs(det) < 1e-12:
            result["diagnostics"]["error"] = f"Affine is singular (det={det})"
            return result
        cond = np.linalg.cond(affine_mat[:, :2])
        best["determinant"] = det
        best["condition_number"] = cond

    # Compute diagnostics
    inlier_mask = best["inlier_mask"]
    n_inliers = best["n_inliers"]
    n_total = len(src_pts)
    inlier_ratio = n_inliers / n_total if n_total > 0 else 0

    # Reprojection errors for inliers
    if best["model"] == "homography":
        src_h = cv2.convertPointsToHomogeneous(src_pts).reshape(-1, 3).T
        proj = best["H"] @ src_h
        proj = proj[:2] / (proj[2] + 1e-12)
        proj = proj.T
    else:
        src_a = np.hstack([src_pts, np.ones((len(src_pts), 1))])
        proj = (best["affine"] @ src_a.T).T

    errors = np.linalg.norm(proj - ref_pts, axis=1)
    inlier_errors = errors[inlier_mask == 1]

    best["diagnostics"] = {
        "n_tentative": n_total,
        "n_inliers": n_inliers,
        "inlier_ratio": float(inlier_ratio),
        "rmse": float(best["rmse"]),
        "median_error": float(np.median(inlier_errors)) if len(inlier_errors) > 0 else None,
        "p95_error": float(np.percentile(inlier_errors, 95)) if len(inlier_errors) > 0 else None,
        "determinant": float(best.get("determinant", 0)),
        "condition_number": float(best.get("condition_number", 0)),
    }

    result["success"] = True
    result["H"] = best["H"] if best["model"] == "homography" else None
    result["affine"] = best["affine"] if best["model"] == "affine" else None
    result["inlier_mask"] = inlier_mask
    result["method"] = best["method"]
    result["model_type"] = best["model"]
    result["diagnostics"] = best["diagnostics"]

    logger.info(f"Geometry verification: {best['model']} via {best['method']}, "
                f"inliers={n_inliers}/{n_total} ({inlier_ratio:.2%}), RMSE={best['rmse']:.3f} px")
    return result

# CELL GROUP J — Grid-Balanced Tie-Point Selection

In [9]:
def grid_balance_matches(src_pts: np.ndarray, ref_pts: np.ndarray, inlier_mask: np.ndarray,
                          image_shape: Tuple[int, int], rows: int, cols: int,
                          max_per_cell: int, errors: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """
    Divide the valid source area into a grid and retain up to max_per_cell best inliers per cell.
    Returns selected indices and coverage score.
    """
    result = {
        "selected_indices": np.array([], dtype=int),
        "coverage_score": 0.0,
        "grid_counts": np.zeros((rows, cols), dtype=int),
        "valid_cells": 0,
        "total_cells": rows * cols,
    }
    if src_pts is None or len(src_pts) < 4:
        return result

    h, w = image_shape[:2]
    cell_h = h / rows
    cell_w = w / cols

    inlier_indices = np.where(inlier_mask == 1)[0]
    if len(inlier_indices) == 0:
        return result

    # Group by cell
    cell_points = {}
    for idx in inlier_indices:
        x, y = src_pts[idx]
        r = min(int(y / cell_h), rows - 1)
        c = min(int(x / cell_w), cols - 1)
        cell_points.setdefault((r, c), []).append(idx)

    selected = []
    for (r, c), indices in cell_points.items():
        # Sort by error (smallest first) or distance
        if errors is not None:
            indices = sorted(indices, key=lambda i: errors[i])
        else:
            indices = sorted(indices, key=lambda i: np.linalg.norm(src_pts[i] - ref_pts[i]))
        selected.extend(indices[:max_per_cell])
        result["grid_counts"][r, c] = min(len(indices), max_per_cell)

    result["selected_indices"] = np.array(selected, dtype=int)
    result["valid_cells"] = len(cell_points)
    result["coverage_score"] = len(cell_points) / (rows * cols) if rows * cols > 0 else 0.0

    logger.info(f"Grid balancing: {len(selected)} selected from {len(inlier_indices)} inliers, "
                f"coverage={result['coverage_score']:.2%} ({len(cell_points)}/{rows*cols} cells)")
    return result


def visualize_grid_matches(source: np.ndarray, reference: np.ndarray,
                            src_pts: np.ndarray, ref_pts: np.ndarray,
                            selected_indices: np.ndarray,
                            grid_counts: np.ndarray,
                            out_dir: Path) -> None:
    """
    Create and save grid match visualizations.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # Source with grid and selected points
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(source, cmap="gray")
    h, w = source.shape[:2]
    rows, cols = grid_counts.shape
    for r in range(rows + 1):
        axes[0].axhline(y=r * h / rows, color="cyan", linewidth=0.5)
    for c in range(cols + 1):
        axes[0].axvline(x=c * w / cols, color="cyan", linewidth=0.5)
    if len(selected_indices) > 0:
        pts = src_pts[selected_indices]
        axes[0].scatter(pts[:, 0], pts[:, 1], c="red", s=15, marker="x")
    axes[0].set_title(f"Source with Grid & Tie Points (n={len(selected_indices)})")
    axes[0].axis("off")

    # Reference with matched points
    axes[1].imshow(reference, cmap="gray")
    if len(selected_indices) > 0:
        pts = ref_pts[selected_indices]
        axes[1].scatter(pts[:, 0], pts[:, 1], c="lime", s=15, marker="x")
    axes[1].set_title("Reference with Matched Points")
    axes[1].axis("off")

    # Side-by-side matches
    h1, w1 = source.shape[:2]
    h2, w2 = reference.shape[:2]
    h_max = max(h1, h2)
    canvas = np.zeros((h_max, w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1, 0] = (source * 255).astype(np.uint8)
    canvas[:h2, w1:w1+w2, 1] = (reference * 255).astype(np.uint8)
    axes[2].imshow(canvas)
    if len(selected_indices) > 0:
        for idx in selected_indices:
            x1, y1 = src_pts[idx]
            x2, y2 = ref_pts[idx]
            axes[2].plot([x1, x2 + w1], [y1, y2], "r-", linewidth=0.5, alpha=0.7)
    axes[2].set_title("Side-by-Side Matches")
    axes[2].axis("off")

    plt.tight_layout()
    plt.savefig(out_dir / "grid_matches.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

    # Heatmap
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    sns.heatmap(grid_counts, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
    ax.set_title("Selected Inliers per Grid Cell")
    plt.tight_layout()
    plt.savefig(out_dir / "grid_heatmap.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    logger.info(f"Saved grid match visualizations to {out_dir}")

# CELL GROUP K — Local Sub-Pixel Refinement

In [10]:
def phase_correlation_refinement(source: np.ndarray, reference: np.ndarray,
                                  src_pts: np.ndarray, ref_pts: np.ndarray,
                                  selected_indices: np.ndarray,
                                  patch_size: int = 41,
                                  response_threshold: float = 0.15,
                                  max_shift: float = 5.0) -> Dict[str, Any]:
    """
    Conservative local sub-pixel refinement using phase correlation.
    Returns refined points and statistics.
    """
    result = {
        "refined_src_pts": [],
        "refined_ref_pts": [],
        "accepted_indices": [],
        "shifts": [],
        "responses": [],
        "attempted": 0,
        "accepted": 0,
    }
    if len(selected_indices) == 0:
        return result

    half = patch_size // 2
    h_src, w_src = source.shape[:2]
    h_ref, w_ref = reference.shape[:2]

    for idx in selected_indices:
        x1, y1 = src_pts[idx]
        x2, y2 = ref_pts[idx]
        result["attempted"] += 1

        # Extract patches
        x1i, y1i = int(round(x1)), int(round(y1))
        x2i, y2i = int(round(x2)), int(round(y2))

        if (x1i - half < 0 or x1i + half >= w_src or y1i - half < 0 or y1i + half >= h_src or
            x2i - half < 0 or x2i + half >= w_ref or y2i - half < 0 or y2i + half >= h_ref):
            continue

        patch1 = source[y1i-half:y1i+half+1, x1i-half:x1i+half+1].astype(np.float32)
        patch2 = reference[y2i-half:y2i+half+1, x2i-half:x2i+half+1].astype(np.float32)

        if patch1.size == 0 or patch2.size == 0:
            continue

        # Apply window to reduce edge effects
        win = cv2.createHanningWindow((patch1.shape[1], patch1.shape[0]), cv2.CV_32F)
        patch1 = patch1 * win
        patch2 = patch2 * win

        try:
            shift, response = cv2.phaseCorrelate(patch1, patch2)
        except Exception as e:
            continue

        if response < response_threshold:
            continue
        shift_mag = np.sqrt(shift[0]**2 + shift[1]**2)
        if shift_mag > max_shift:
            continue

        # Refined reference point
        refined_x = x2 + shift[0]
        refined_y = y2 + shift[1]

        result["refined_src_pts"].append([x1, y1])
        result["refined_ref_pts"].append([refined_x, refined_y])
        result["accepted_indices"].append(idx)
        result["shifts"].append(shift_mag)
        result["responses"].append(response)
        result["accepted"] += 1

    result["refined_src_pts"] = np.array(result["refined_src_pts"]) if result["refined_src_pts"] else np.array([])
    result["refined_ref_pts"] = np.array(result["refined_ref_pts"]) if result["refined_ref_pts"] else np.array([])
    result["accepted_indices"] = np.array(result["accepted_indices"], dtype=int) if result["accepted_indices"] else np.array([], dtype=int)

    logger.info(f"Phase correlation refinement: attempted={result['attempted']}, accepted={result['accepted']}")
    if result["shifts"]:
        logger.info(f"  Median shift: {np.median(result['shifts']):.3f} px, "
                    f"Response stats: min={min(result['responses']):.3f}, max={max(result['responses']):.3f}")
    return result


# CELL GROUP L — Registration Confidence Score

In [11]:
def compute_registration_confidence_score(verification: Dict[str, Any],
                                           grid_result: Dict[str, Any],
                                           refinement: Dict[str, Any],
                                           config: dict,
                                           illumination_validity: float = 0.5) -> Dict[str, Any]:
    """
    Compute a transparent Registration Confidence Score (0-100).
    """
    result = {
        "rcs": 0.0,
        "components": {},
        "decision": "REJECT",
        "safety_gates": {},
        "warnings": [],
    }

    if not verification.get("success", False):
        result["warnings"].append("Geometry verification failed")
        result["safety_gates"]["valid_transform"] = False
        return result

    diag = verification["diagnostics"]
    n_inliers = diag["n_inliers"]
    inlier_ratio = diag["inlier_ratio"]
    rmse = diag["rmse"]

    # Component scores
    # Residual score
    tau_rmse = max(1.0, config["ransac_reproj_threshold"])
    residual_score = np.exp(-rmse / tau_rmse) if np.isfinite(rmse) else 0.0
    residual_score = np.clip(residual_score, 0, 1)

    # Inlier score
    inlier_score = 0.6 * inlier_ratio + 0.4 * min(1.0, n_inliers / 50.0)
    inlier_score = np.clip(inlier_score, 0, 1)

    # Coverage score
    coverage_score = grid_result.get("coverage_score", 0.0)
    coverage_score = np.clip(coverage_score, 0, 1)

    # Stability score (bootstrap)
    stability_score = 0.0
    if n_inliers >= 12:
        # Bootstrap resampling
        n_bootstrap = 10
        displacements = []
        src_pts = verification.get("src_pts", None)
        ref_pts = verification.get("ref_pts", None)
        if src_pts is not None and ref_pts is not None and len(src_pts) >= 12:
            for _ in range(n_bootstrap):
                indices = np.random.choice(len(src_pts), size=int(0.8 * len(src_pts)), replace=False)
                try:
                    H_boot, _ = cv2.findHomography(src_pts[indices], ref_pts[indices], cv2.RANSAC, 3.0)
                    if H_boot is not None:
                        # Test at 5x5 grid
                        test_pts = np.array([[i * 100, j * 100] for i in range(5) for j in range(5)], dtype=np.float32)
                        test_h = cv2.convertPointsToHomogeneous(test_pts).reshape(-1, 3).T
                        proj = H_boot @ test_h
                        proj = proj[:2] / (proj[2] + 1e-12)
                        proj = proj.T
                        # Compare to original H
                        H_orig = verification.get("H") if verification["model_type"] == "homography" else None
                        if H_orig is not None:
                            proj_orig = H_orig @ test_h
                            proj_orig = proj_orig[:2] / (proj_orig[2] + 1e-12)
                            proj_orig = proj_orig.T
                            disp = np.linalg.norm(proj - proj_orig, axis=1)
                            displacements.extend(disp)
                except Exception:
                    pass
        if displacements:
            median_disp = np.median(displacements)
            stability_score = np.exp(-median_disp / 3.0)
        else:
            stability_score = 0.1  # Conservative low score
            result["warnings"].append("Stability test could not be completed; conservative low score assigned")
    else:
        stability_score = 0.1
        result["warnings"].append(f"Only {n_inliers} inliers; insufficient for stability test (need >= 12)")
    stability_score = np.clip(stability_score, 0, 1)

    # Illumination validity score
    illumination_score = np.clip(illumination_validity, 0, 1)

    # RCS
    rcs = 100 * (
        0.30 * residual_score +
        0.25 * inlier_score +
        0.20 * coverage_score +
        0.15 * stability_score +
        0.10 * illumination_score
    )
    rcs = float(np.clip(rcs, 0, 100))

    result["rcs"] = rcs
    result["components"] = {
        "residual_score": float(residual_score),
        "inlier_score": float(inlier_score),
        "coverage_score": float(coverage_score),
        "stability_score": float(stability_score),
        "illumination_validity_score": float(illumination_score),
    }

    # Safety gates
    result["safety_gates"] = {
        "min_inliers": n_inliers >= config["min_inliers"],
        "min_inlier_ratio": inlier_ratio >= config["min_inlier_ratio"],
        "min_coverage": coverage_score >= config["min_coverage"],
        "valid_transform": True,
        "finite_rmse": np.isfinite(rmse),
    }

    # Decision
    all_gates_pass = all(result["safety_gates"].values())
    if rcs >= config["accept_rcs"] and all_gates_pass:
        result["decision"] = "ACCEPT"
    elif rcs >= config["review_rcs"]:
        result["decision"] = "REVIEW"
    else:
        result["decision"] = "REJECT"

    # Add warnings for failed gates
    for gate, passed in result["safety_gates"].items():
        if not passed:
            result["warnings"].append(f"Safety gate failed: {gate}")

    logger.info(f"RCS: {rcs:.1f}/100, Decision: {result['decision']}, "
                f"Components: {result['components']}")
    return result


# CELL GROUP M — Register Image and Generate Deliverables

In [12]:
def register_image(source: np.ndarray, reference: np.ndarray,
                    transform: Dict[str, Any],
                    out_dir: Path,
                    source_sensor: str, reference_sensor: str) -> Dict[str, Any]:
    """
    Warp source to reference frame and generate deliverables.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    result = {}

    h_ref, w_ref = reference.shape[:2]

    if transform.get("H") is not None:
        H = transform["H"]
        warped = cv2.warpPerspective(source, H, (w_ref, h_ref), flags=cv2.INTER_LINEAR)
    elif transform.get("affine") is not None:
        affine = transform["affine"]
        warped = cv2.warpAffine(source, affine, (w_ref, h_ref), flags=cv2.INTER_LINEAR)
    else:
        logger.error("No valid transform found")
        return result

    # Save registered grayscale
    cv2.imwrite(str(out_dir / "registered_grayscale.png"), (warped * 255).astype(np.uint8))

    # Alpha blend
    alpha = 0.5
    overlay = (alpha * warped + (1 - alpha) * reference)
    overlay = np.clip(overlay, 0, 1)
    cv2.imwrite(str(out_dir / "overlay_alpha.png"), (overlay * 255).astype(np.uint8))

    # Checkerboard
    checker = np.zeros_like(reference)
    tile = 64
    for i in range(0, h_ref, tile):
        for j in range(0, w_ref, tile):
            if ((i // tile) + (j // tile)) % 2 == 0:
                checker[i:i+tile, j:j+tile] = warped[i:i+tile, j:j+tile]
            else:
                checker[i:i+tile, j:j+tile] = reference[i:i+tile, j:j+tile]
    cv2.imwrite(str(out_dir / "checkerboard.png"), (checker * 255).astype(np.uint8))

    # Absolute difference
    diff = np.abs(warped - reference)
    cv2.imwrite(str(out_dir / "absolute_difference.png"), normalize_to_uint8(diff))

    # Side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(reference, cmap="gray")
    axes[0].set_title("Reference")
    axes[0].axis("off")
    axes[1].imshow(warped, cmap="gray")
    axes[1].set_title("Registered Source")
    axes[1].axis("off")
    axes[2].imshow(overlay, cmap="gray")
    axes[2].set_title("Overlay")
    axes[2].axis("off")
    plt.tight_layout()
    plt.savefig(out_dir / "before_after_comparison.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

    result["warped"] = warped
    result["overlay"] = overlay
    logger.info(f"Saved registered outputs to {out_dir}")
    return result

# CELL GROUP N — HTML/PDF-Style Report

In [13]:
def generate_html_report(experiment_name: str,
                          source_sensor: str, reference_sensor: str,
                          source_path: str, reference_path: str,
                          verification: Dict[str, Any],
                          grid_result: Dict[str, Any],
                          refinement: Dict[str, Any],
                          rcs_result: Dict[str, Any],
                          output_dir: Path,
                          config: dict) -> Path:
    """
    Generate a human-readable HTML report.
    """
    report_path = output_dir / "registration_report.html"

    diag = verification.get("diagnostics", {})
    comp = rcs_result.get("components", {})

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>LunarXReg Report — {experiment_name}</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 20px; background: #f5f5f5; }}
h1 {{ color: #1a237e; }}
h2 {{ color: #283593; border-bottom: 2px solid #283593; padding-bottom: 4px; }}
table {{ border-collapse: collapse; width: 100%; margin: 10px 0; background: white; }}
th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; }}
th {{ background: #e8eaf6; }}
.status-accept {{ color: green; font-weight: bold; }}
.status-review {{ color: orange; font-weight: bold; }}
.status-reject {{ color: red; font-weight: bold; }}
.warning {{ background: #fff3cd; border-left: 4px solid #ffc107; padding: 10px; margin: 10px 0; }}
</style>
</head>
<body>
<h1>LunarXReg: Registration Report</h1>
<p><strong>Experiment:</strong> {experiment_name}</p>
<p><strong>Source Sensor:</strong> {source_sensor} — {source_path}</p>
<p><strong>Reference Sensor:</strong> {reference_sensor} — {reference_path}</p>
<p><strong>Mode:</strong> {config['mode']}</p>
<p><strong>Timestamp:</strong> {datetime.now().isoformat()}</p>

<h2>Registration Confidence Score</h2>
<table>
<tr><th>Metric</th><th>Value</th><th>Threshold</th><th>Status</th></tr>
<tr><td>RCS</td><td>{rcs_result['rcs']:.1f}</td><td>ACCEPT >= {config['accept_rcs']}, REVIEW >= {config['review_rcs']}</td><td class="status-{rcs_result['decision'].lower()}">{rcs_result['decision']}</td></tr>
<tr><td>Inliers</td><td>{diag.get('n_inliers', 0)}</td><td>>= {config['min_inliers']}</td><td>{'PASS' if rcs_result['safety_gates'].get('min_inliers') else 'FAIL'}</td></tr>
<tr><td>Inlier Ratio</td><td>{diag.get('inlier_ratio', 0):.3f}</td><td>>= {config['min_inlier_ratio']}</td><td>{'PASS' if rcs_result['safety_gates'].get('min_inlier_ratio') else 'FAIL'}</td></tr>
<tr><td>Coverage</td><td>{grid_result.get('coverage_score', 0):.3f}</td><td>>= {config['min_coverage']}</td><td>{'PASS' if rcs_result['safety_gates'].get('min_coverage') else 'FAIL'}</td></tr>
<tr><td>RMSE</td><td>{diag.get('rmse', 'N/A')}</td><td>finite</td><td>{'PASS' if rcs_result['safety_gates'].get('finite_rmse') else 'FAIL'}</td></tr>
</table>

<h2>RCS Components</h2>
<table>
<tr><th>Component</th><th>Score (0-1)</th><th>Weight</th><th>Contribution</th></tr>
<tr><td>Residual</td><td>{comp.get('residual_score', 0):.3f}</td><td>0.30</td><td>{0.30*comp.get('residual_score', 0):.3f}</td></tr>
<tr><td>Inlier</td><td>{comp.get('inlier_score', 0):.3f}</td><td>0.25</td><td>{0.25*comp.get('inlier_score', 0):.3f}</td></tr>
<tr><td>Coverage</td><td>{comp.get('coverage_score', 0):.3f}</td><td>0.20</td><td>{0.20*comp.get('coverage_score', 0):.3f}</td></tr>
<tr><td>Stability</td><td>{comp.get('stability_score', 0):.3f}</td><td>0.15</td><td>{0.15*comp.get('stability_score', 0):.3f}</td></tr>
<tr><td>Illumination Validity</td><td>{comp.get('illumination_validity_score', 0):.3f}</td><td>0.10</td><td>{0.10*comp.get('illumination_validity_score', 0):.3f}</td></tr>
</table>

<h2>Geometry Verification</h2>
<table>
<tr><th>Metric</th><th>Value</th></tr>
<tr><td>Transform Model</td><td>{verification.get('model_type', 'N/A')}</td></tr>
<tr><td>Method</td><td>{verification.get('method', 'N/A')}</td></tr>
<tr><td>Tentative Matches</td><td>{diag.get('n_tentative', 0)}</td></tr>
<tr><td>Inliers</td><td>{diag.get('n_inliers', 0)}</td></tr>
<tr><td>Inlier Ratio</td><td>{diag.get('inlier_ratio', 0):.3f}</td></tr>
<tr><td>RMSE</td><td>{diag.get('rmse', 'N/A')}</td></tr>
<tr><td>Median Error</td><td>{diag.get('median_error', 'N/A')}</td></tr>
<tr><td>95th Percentile Error</td><td>{diag.get('p95_error', 'N/A')}</td></tr>
<tr><td>Determinant</td><td>{diag.get('determinant', 'N/A')}</td></tr>
<tr><td>Condition Number</td><td>{diag.get('condition_number', 'N/A')}</td></tr>
</table>

<h2>Sub-Pixel Refinement</h2>
<table>
<tr><th>Metric</th><th>Value</th></tr>
<tr><td>Attempted</td><td>{refinement.get('attempted', 0)}</td></tr>
<tr><td>Accepted</td><td>{refinement.get('accepted', 0)}</td></tr>
<tr><td>Median Shift (px)</td><td>{np.median(refinement['shifts']) if refinement.get('shifts') else 'N/A'}</td></tr>
<tr><td>Response Stats</td><td>min={min(refinement['responses']) if refinement.get('responses') else 'N/A'}, max={max(refinement['responses']) if refinement.get('responses') else 'N/A'}</td></tr>
</table>

<h2>Grid Balance</h2>
<table>
<tr><th>Metric</th><th>Value</th></tr>
<tr><td>Valid Cells</td><td>{grid_result.get('valid_cells', 0)}</td></tr>
<tr><td>Total Cells</td><td>{grid_result.get('total_cells', 0)}</td></tr>
<tr><td>Coverage Score</td><td>{grid_result.get('coverage_score', 0):.3f}</td></tr>
<tr><td>Selected Tie Points</td><td>{len(grid_result.get('selected_indices', []))}</td></tr>
</table>

<h2>Warnings and Limitations</h2>
<div class="warning">
<ul>
{''.join(f'<li>{w}</li>' for w in rcs_result.get('warnings', []))}
<li>This report does not claim mission-grade or guaranteed sub-pixel accuracy. Only measured residuals are reported.</li>
<li>Real registration success requires physical overlap. Non-overlapping images will yield REVIEW or REJECT.</li>
<li>GeoTIFF is produced only if valid CRS/geotransform is available; otherwise outputs are in image space.</li>
</ul>
</div>

<h2>Diagnostic Images</h2>
<p>See output directory: {output_dir}</p>
<ul>
<li>grid_matches.png</li>
<li>grid_heatmap.png</li>
<li>registered_grayscale.png</li>
<li>overlay_alpha.png</li>
<li>checkerboard.png</li>
<li>absolute_difference.png</li>
<li>before_after_comparison.png</li>
</ul>

</body>
</html>
"""
    with open(report_path, "w") as f:
        f.write(html)
    logger.info(f"Saved HTML report: {report_path}")
    return report_path


def append_experiment_summary(experiment_name: str, source_sensor: str, reference_sensor: str,
                               rcs_result: Dict[str, Any], verification: Dict[str, Any],
                               grid_result: Dict[str, Any]) -> None:
    """Append a summary row to experiment_summary.csv."""
    summary_path = RESULTS_DIR / "reports" / "experiment_summary.csv"
    diag = verification.get("diagnostics", {})
    row = {
        "timestamp": datetime.now().isoformat(),
        "experiment": experiment_name,
        "source_sensor": source_sensor,
        "reference_sensor": reference_sensor,
        "decision": rcs_result.get("decision", "REJECT"),
        "rcs": rcs_result.get("rcs", 0),
        "n_inliers": diag.get("n_inliers", 0),
        "n_tentative": diag.get("n_tentative", 0),
        "inlier_ratio": diag.get("inlier_ratio", 0),
        "rmse": diag.get("rmse", None),
        "coverage": grid_result.get("coverage_score", 0),
        "model_type": verification.get("model_type", None),
        "method": verification.get("method", None),
    }
    df = pd.DataFrame([row])
    if summary_path.exists():
        df.to_csv(summary_path, mode="a", header=False, index=False)
    else:
        df.to_csv(summary_path, index=False)
    logger.info(f"Appended summary to {summary_path}")


# CELL GROUP O — Synthetic Validation Mode

In [14]:

def run_synthetic_validation(config: dict) -> Dict[str, Any]:
    """
    Run the full pipeline on a known synthetic transformation.
    """
    print("\n" + "=" * 60)
    print("SYNTHETIC TRANSFORMATION VALIDATION")
    print("This validates the registration engine on known transformations.")
    print("Not an inter-sensor scientific accuracy claim.")
    print("=" * 60)

    out_dir = RESULTS_DIR / "synthetic_validation" / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load source sensor
    sensor = config["synthetic_source_sensor"]
    data = sensor_data.get(sensor)

# Auto-fallback: if preferred sensor has a thin/strip image, try other sensors
    def _is_good_for_synth(d):
        if d is None or d["raw_image"] is None:
            return False
        h, w = d["raw_image"].shape[:2]
        return min(h, w) >= 300   # needs some 2D content

    if not _is_good_for_synth(data):
        logger.warning(f"Preferred synthetic source '{sensor}' has insufficient 2D content "
                   f"({data['raw_image'].shape if data and data['raw_image'] is not None else 'None'}). "
                   f"Falling back to another sensor.")
        for alt in ["ohrc", "tmc2", "iirs"]:
            if _is_good_for_synth(sensor_data.get(alt)):
                sensor = alt
                data = sensor_data[alt]
                logger.info(f"Using '{sensor}' for synthetic validation instead.")
                break
    if data is None or data["raw_image"] is None:
        logger.error(f"Cannot run synthetic validation: {sensor} image not loaded")
        print(f"ERROR: {sensor} image not loaded. Cannot run synthetic validation.")
        return {"success": False, "error": f"{sensor} image not loaded"}

    source_img = data["raw_image"].astype(np.float32)
    if source_img.max() > 1.0:
        source_img = source_img / 255.0

    # Create synthetic reference by applying known transform
    h, w = source_img.shape[:2]
    center = (w / 2, h / 2)
    angle = config["synthetic_transform"]["rotation_deg"]
    scale = config["synthetic_transform"]["scale"]
    tx = config["synthetic_transform"]["tx_px"]
    ty = config["synthetic_transform"]["ty_px"]
    persp_jitter = config["synthetic_transform"]["perspective_jitter_px"]

    # Affine matrix
    M = cv2.getRotationMatrix2D(center, angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty

    # Perspective perturbation
    if persp_jitter > 0:
        src_pts = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
        dst_pts = src_pts + np.random.uniform(-persp_jitter, persp_jitter, src_pts.shape).astype(np.float32)
        H_persp = cv2.getPerspectiveTransform(src_pts, dst_pts)
        # Combine: first affine, then perspective
        M_h = np.vstack([M, [0, 0, 1]])
        H_combined = H_persp @ M_h
    else:
        H_combined = np.vstack([M, [0, 0, 1]])

    # Warp
    synthetic_ref = cv2.warpPerspective(source_img, H_combined, (w, h), flags=cv2.INTER_LINEAR)
    # Add mild illumination variation
    illumination = np.linspace(0.8, 1.2, w).reshape(1, -1)
    synthetic_ref = synthetic_ref * illumination
    synthetic_ref = np.clip(synthetic_ref, 0, 1)

    # Ground truth transform
    gt_transform = {"H": H_combined.tolist(), "type": "homography"}

    # Save synthetic pair preview
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(source_img, cmap="gray")
    axes[0].set_title("Source (Original)")
    axes[0].axis("off")
    axes[1].imshow(synthetic_ref, cmap="gray")
    axes[1].set_title("Synthetic Reference (Transformed)")
    axes[1].axis("off")
    plt.suptitle("Synthetic Transformation Validation")
    plt.tight_layout()
    plt.savefig(out_dir / "synthetic_pair.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

    # Create structural views
    src_struct = create_structural_view(source_img, f"{sensor}_synthetic_source", config)
    ref_struct = create_structural_view(synthetic_ref, f"{sensor}_synthetic_ref", config)

    if not src_struct or not ref_struct:
        return {"success": False, "error": "Structural preprocessing failed"}

    # Working resolution
    res_info = choose_working_resolution(src_struct["final"], ref_struct["final"], config["max_working_dimension"])
    src_work = res_info["source_working"]
    ref_work = res_info["reference_working"]
    src_mask = cv2.resize(src_struct["shadow_mask"], (src_work.shape[1], src_work.shape[0]), interpolation=cv2.INTER_NEAREST)
    ref_mask = cv2.resize(ref_struct["shadow_mask"], (ref_work.shape[1], ref_work.shape[0]), interpolation=cv2.INTER_NEAREST)

    # Matcher cascade
    print("\nRunning matcher cascade on synthetic pair...")
    matcher_results = run_adaptive_matcher_cascade(src_work, ref_work, src_mask, ref_mask, config)

    # Select best matcher
    best_result = None
    best_rcs = -1
    all_verifications = []
    for mr in matcher_results:
        if mr["raw_match_count"] < 4:
            continue
        ver = fit_transform_and_verify(mr["keypoints_source"], mr["keypoints_reference"], config)
        if not ver["success"]:
            continue
        # Provisional RCS for selection
        grid = grid_balance_matches(mr["keypoints_source"], mr["keypoints_reference"],
                                     ver["inlier_mask"], src_work.shape,
                                     config["n_grid_rows"], config["n_grid_cols"],
                                     config["max_matches_per_cell"])
        rcs = compute_registration_confidence_score(ver, grid, {}, config)
        all_verifications.append({"matcher": mr["name"], "verification": ver, "grid": grid, "rcs": rcs})
        if rcs["rcs"] > best_rcs:
            best_rcs = rcs["rcs"]
            best_result = {"matcher": mr, "verification": ver, "grid": grid, "rcs": rcs}

    if best_result is None:
        print("No valid registration found for synthetic pair.")
        return {"success": False, "error": "No valid registration"}

    mr = best_result["matcher"]
    ver = best_result["verification"]
    grid = best_result["grid"]
    rcs_result = best_result["rcs"]

    print(f"\nSelected matcher: {mr['name']}")
    print(f"RCS: {rcs_result['rcs']:.1f}/100, Decision: {rcs_result['decision']}")

    # Compare estimated vs ground truth
    if ver["H"] is not None:
        estimated_H = ver["H"]
        gt_H = np.array(gt_transform["H"])
        # Corner reprojection error
        corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
        gt_proj = cv2.perspectiveTransform(corners.reshape(-1, 1, 2), gt_H).reshape(-1, 2)
        est_proj = cv2.perspectiveTransform(corners.reshape(-1, 1, 2), estimated_H).reshape(-1, 2)
        corner_errors = np.linalg.norm(gt_proj - est_proj, axis=1)
        center_error = np.linalg.norm(np.mean(gt_proj, axis=0) - np.mean(est_proj, axis=0))

        print("\n=== SYNTHETIC VALIDATION RESULTS ===")
        print(f"Ground truth transform type: {gt_transform['type']}")
        print(f"Estimated transform type: {ver['model_type']}")
        print(f"Corner reprojection errors (px): {corner_errors}")
        print(f"Mean corner error: {np.mean(corner_errors):.3f} px")
        print(f"Center displacement error: {center_error:.3f} px")

        validation_table = pd.DataFrame({
            "Corner": [0, 1, 2, 3],
            "GT_x": gt_proj[:, 0], "GT_y": gt_proj[:, 1],
            "Est_x": est_proj[:, 0], "Est_y": est_proj[:, 1],
            "Error_px": corner_errors,
        })
        validation_table.to_csv(out_dir / "validation_corner_errors.csv", index=False)
        print("\n", validation_table.to_string())

    # Save results
    with open(out_dir / "rcs_result.json", "w") as f:
        json.dump(rcs_result, f, indent=2, default=str)
    with open(out_dir / "verification.json", "w") as f:
        json.dump({k: v for k, v in ver.items() if k != "inlier_mask"}, f, indent=2, default=str)

    # Generate report
    generate_html_report("synthetic_validation", sensor, "synthetic_reference",
                          data.get("raw_path", "unknown"), "synthetic_generated",
                          ver, grid, {}, rcs_result, out_dir, config)

    return {
        "success": True,
        "rcs_result": rcs_result,
        "verification": ver,
        "grid": grid,
        "matcher": mr,
        "output_dir": str(out_dir),
    }

# CELL GROUP P — Optional Real Pair Experiments

In [15]:
def run_real_pair_experiment(source_sensor: str, reference_sensor: str, config: dict) -> Dict[str, Any]:
    """
    Run registration for a real source-reference pair.
    """
    print(f"\n{'='*60}")
    print(f"REAL PAIR EXPERIMENT: {source_sensor} -> {reference_sensor}")
    print(f"{'='*60}")

    src_data = sensor_data.get(source_sensor)
    ref_data = sensor_data.get(reference_sensor)

    if src_data is None or src_data["raw_image"] is None:
        logger.error(f"Source sensor {source_sensor} not loaded")
        print(f"ERROR: Source sensor {source_sensor} not loaded.")
        return {"success": False, "error": f"{source_sensor} not loaded"}

    if ref_data is None or ref_data["raw_image"] is None:
        logger.error(f"Reference sensor {reference_sensor} not loaded")
        print(f"ERROR: Reference sensor {reference_sensor} not loaded.")
        return {"success": False, "error": f"{reference_sensor} not loaded"}

    out_dir = RESULTS_DIR / f"{source_sensor}_to_{reference_sensor}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=True)

    source_img = src_data["raw_image"].astype(np.float32)
    if source_img.max() > 1.0:
        source_img = source_img / 255.0
    ref_img = ref_data["raw_image"].astype(np.float32)
    if ref_img.max() > 1.0:
        ref_img = ref_img / 255.0

    # Manual ROI if provided
    if config["manual_roi_source"]:
        source_img = crop_roi(source_img, config["manual_roi_source"])
    if config["manual_roi_reference"]:
        ref_img = crop_roi(ref_img, config["manual_roi_reference"])

    # Structural views
    src_struct = create_structural_view(source_img, f"{source_sensor}_real", config)
    ref_struct = create_structural_view(ref_img, f"{reference_sensor}_real", config)

    if not src_struct or not ref_struct:
        return {"success": False, "error": "Structural preprocessing failed"}

    # Working resolution
    res_info = choose_working_resolution(src_struct["final"], ref_struct["final"], config["max_working_dimension"])
    src_work = res_info["source_working"]
    ref_work = res_info["reference_working"]
    src_mask = cv2.resize(src_struct["shadow_mask"], (src_work.shape[1], src_work.shape[0]), interpolation=cv2.INTER_NEAREST)
    ref_mask = cv2.resize(ref_struct["shadow_mask"], (ref_work.shape[1], ref_work.shape[0]), interpolation=cv2.INTER_NEAREST)

    print(f"Working resolution: source {src_work.shape}, reference {ref_work.shape}")

    # Matcher cascade
    print("Running matcher cascade...")
    matcher_results = run_adaptive_matcher_cascade(src_work, ref_work, src_mask, ref_mask, config)

    # Evaluate all matchers
    best_result = None
    best_rcs = -1
    matcher_summary = []

    for mr in matcher_results:
        if mr["raw_match_count"] < 4:
            matcher_summary.append({
                "matcher": mr["name"],
                "raw_matches": mr["raw_match_count"],
                "status": "insufficient matches",
                "n_inliers": 0,
                "rcs": 0,
            })
            continue
        ver = fit_transform_and_verify(mr["keypoints_source"], mr["keypoints_reference"], config)
        if not ver["success"]:
            matcher_summary.append({
                "matcher": mr["name"],
                "raw_matches": mr["raw_match_count"],
                "status": "verification failed",
                "n_inliers": 0,
                "rcs": 0,
            })
            continue

        grid = grid_balance_matches(mr["keypoints_source"], mr["keypoints_reference"],
                                     ver["inlier_mask"], src_work.shape,
                                     config["n_grid_rows"], config["n_grid_cols"],
                                     config["max_matches_per_cell"])
        rcs = compute_registration_confidence_score(ver, grid, {}, config)
        matcher_summary.append({
            "matcher": mr["name"],
            "raw_matches": mr["raw_match_count"],
            "status": "ok",
            "n_inliers": ver["diagnostics"]["n_inliers"],
            "rcs": rcs["rcs"],
            "decision": rcs["decision"],
        })
        if rcs["rcs"] > best_rcs:
            best_rcs = rcs["rcs"]
            best_result = {"matcher": mr, "verification": ver, "grid": grid, "rcs": rcs}

    matcher_df = pd.DataFrame(matcher_summary)
    print("\nMatcher Summary:")
    print(matcher_df.to_string())
    matcher_df.to_csv(out_dir / "matcher_summary.csv", index=False)

    if best_result is None:
        print(f"\nResult: REJECT — No valid registration found for {source_sensor} -> {reference_sensor}")
        print("Recommended next action: choose overlapping ROI, use a geometrically closer reference, or run synthetic validation.")
        return {"success": False, "decision": "REJECT", "output_dir": str(out_dir)}

    mr = best_result["matcher"]
    ver = best_result["verification"]
    grid = best_result["grid"]
    rcs_result = best_result["rcs"]

    print(f"\nSelected matcher: {mr['name']} (highest RCS, not raw match count)")
    print(f"Verified inliers: {ver['diagnostics']['n_inliers']}")
    print(f"RMSE: {ver['diagnostics']['rmse']:.3f} px")
    print(f"Coverage: {grid['coverage_score']:.3f}")
    print(f"RCS: {rcs_result['rcs']:.1f}/100")
    print(f"Decision: {rcs_result['decision']}")

    # Visualize grid matches
    visualize_grid_matches(src_work, ref_work, mr["keypoints_source"], mr["keypoints_reference"],
                           grid["selected_indices"], grid["grid_counts"], out_dir / "matches")

    # Phase refinement
    print("\nRunning local sub-pixel refinement...")
    refinement = phase_correlation_refinement(src_work, ref_work,
                                               mr["keypoints_source"], mr["keypoints_reference"],
                                               grid["selected_indices"])
    print(f"Refinement: attempted={refinement['attempted']}, accepted={refinement['accepted']}")

    # Register image
    print("\nGenerating registered output...")
    reg_result = register_image(src_work, ref_work, ver, out_dir / "registered",
                                 source_sensor, reference_sensor)

    # Save match CSV
    if len(grid["selected_indices"]) > 0:
        indices = grid["selected_indices"]
        match_data = {
            "source_x_working": mr["keypoints_source"][indices, 0],
            "source_y_working": mr["keypoints_source"][indices, 1],
            "reference_x_working": mr["keypoints_reference"][indices, 0],
            "reference_y_working": mr["keypoints_reference"][indices, 1],
            "source_x_original": mr["keypoints_source"][indices, 0] * res_info["source_scale_x"],
            "source_y_original": mr["keypoints_source"][indices, 1] * res_info["source_scale_y"],
            "reference_x_original": mr["keypoints_reference"][indices, 0] * res_info["reference_scale_x"],
            "reference_y_original": mr["keypoints_reference"][indices, 1] * res_info["reference_scale_y"],
            "reprojection_error_px": ver["diagnostics"].get("rmse", 0),
            "phase_response": 0.0,
            "shadow_valid_source": 1,
            "shadow_valid_reference": 1,
            "selected_after_grid_balance": 1,
        }
        match_df = pd.DataFrame(match_data)
        match_df.to_csv(out_dir / "matches" / "tie_points.csv", index=False)
        print(f"Saved {len(match_df)} tie points to CSV")

    # Save transform
    with open(out_dir / "transform.json", "w") as f:
        json.dump({
            "model_type": ver["model_type"],
            "method": ver["method"],
            "H": ver["H"].tolist() if ver["H"] is not None else None,
            "affine": ver["affine"].tolist() if ver["affine"] is not None else None,
        }, f, indent=2)

    # Generate report
    generate_html_report(f"{source_sensor}_to_{reference_sensor}",
                          source_sensor, reference_sensor,
                          src_data.get("raw_path", "unknown"),
                          ref_data.get("raw_path", "unknown"),
                          ver, grid, refinement, rcs_result, out_dir, config)

    append_experiment_summary(f"{source_sensor}_to_{reference_sensor}",
                               source_sensor, reference_sensor,
                               rcs_result, ver, grid)

    return {
        "success": True,
        "decision": rcs_result["decision"],
        "rcs": rcs_result["rcs"],
        "matcher": mr["name"],
        "n_inliers": ver["diagnostics"]["n_inliers"],
        "rmse": ver["diagnostics"]["rmse"],
        "coverage": grid["coverage_score"],
        "output_dir": str(out_dir),
    }

# CELL GROUP Q — Final Demo Dashboard

In [16]:

def print_dashboard(result: Dict[str, Any]) -> None:
    """Print a presenter-ready dashboard summary."""
    print("\n" + "=" * 70)
    print("LunarXReg — FINAL DEMO DASHBOARD")
    print("=" * 70)

    if not result.get("success", False):
        print(f"Status: {result.get('decision', 'REJECT')}")
        print(f"Reason: {result.get('error', 'Unknown')}")
        print("\nNo reliable registration was forced.")
        print("Recommended next action: choose overlapping ROI, use a geometrically closer reference, or run synthetic validation.")
        return

    print(f"Source → Reference: {result.get('source_sensor', '?')} → {result.get('reference_sensor', '?')}")
    print(f"Selected Matcher: {result.get('matcher', '?')}")
    print(f"Verified Inliers: {result.get('n_inliers', 0)}")
    print(f"RMSE: {result.get('rmse', 0):.3f} px")
    print(f"Coverage: {result.get('coverage', 0):.3f}")
    print(f"RCS: {result.get('rcs', 0):.1f}/100")
    print(f"Decision: {result.get('decision', 'REJECT')}")
    print(f"Output Directory: {result.get('output_dir', 'N/A')}")

    if result.get("decision") == "REJECT":
        print("\nNo reliable registration was forced.")
        print("Recommended next action: choose overlapping ROI, use a geometrically closer reference, or run synthetic validation.")

    print("\nOutput files:")
    out_dir = Path(result.get("output_dir", ""))
    if out_dir.exists():
        for f in sorted(out_dir.rglob("*")):
            if f.is_file():
                print(f"  {f}")

In [17]:

# ============================================================
# MAIN EXECUTION
# ============================================================

print("\n" + "=" * 70)
print("LunarXReg: Chandrayaan-2 Multi-Modal Image Registration MVP")
print("=" * 70)
print(f"Mode: {CONFIG['mode']}")
print(f"Source sensor: {CONFIG['source_sensor']}")
print(f"Reference sensor: {CONFIG['reference_sensor']}")
print(f"GPU available: {GPU_AVAILABLE}")

final_result = None

if CONFIG["mode"] == "synthetic_validation":
    final_result = run_synthetic_validation(CONFIG)
elif CONFIG["mode"] == "real_pair":
    final_result = run_real_pair_experiment(
        CONFIG["source_sensor"],
        CONFIG["reference_sensor"],
        CONFIG
    )
else:
    print(f"Unknown mode: {CONFIG['mode']}. Valid modes: 'real_pair', 'synthetic_validation'")
    final_result = {"success": False, "error": f"Unknown mode: {CONFIG['mode']}"}

# Print dashboard
print_dashboard(final_result)

# Save final result summary
with open(RESULTS_DIR / "reports" / "final_result.json", "w") as f:
    json.dump(final_result, f, indent=2, default=str)

print("\n" + "=" * 70)
print("LunarXReg execution complete.")
print(f"All outputs saved under: {RESULTS_DIR}")
print("=" * 70)


LunarXReg: Chandrayaan-2 Multi-Modal Image Registration MVP
Mode: synthetic_validation
Source sensor: tmc2
Reference sensor: ohrc
GPU available: True

SYNTHETIC TRANSFORMATION VALIDATION
This validates the registration engine on known transformations.
Not an inter-sensor scientific accuracy claim.

Running matcher cascade on synthetic pair...
Running SIFT matcher...
Running ORB matcher...


Running AKAZE matcher...
No valid registration found for synthetic pair.

LunarXReg — FINAL DEMO DASHBOARD
Status: REJECT
Reason: No valid registration

No reliable registration was forced.
Recommended next action: choose overlapping ROI, use a geometrically closer reference, or run synthetic validation.

LunarXReg execution complete.
All outputs saved under: /content/drive/MyDrive/26166_DATA/results
